<a href="https://colab.research.google.com/github/farrelrassya/LLM-Design-Patterns-Building-Robust-and-Efficient-AI-Systems/blob/main/13.Quantization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 13 -- Quantization

*Companion notebook untuk* LLM Design Patterns
**Technical edition -- kode buku plus beberapa pengukuran tambahan**

GPT-2 versi terkecil punya sekitar 124 juta parameter, dan setiap parameter disimpan sebagai float32: empat byte per angka, hampir 500 MB untuk seluruh model. *Quantization* menawarkan jalan pintas yang menggoda. Simpan setiap angka dengan bit yang lebih sedikit, misalnya int8 yang hanya satu byte, maka modelnya tinggal sekitar seperempat; dan menghitung dengan bilangan bulat kecil biasanya lebih cepat daripada dengan float. Buku merangkum janji itu di Table 13.1: untuk PTQ (*post-training quantization*), penghematan penyimpanan 75% dan percepatan 2-4x.

Tentu ada harganya. Angka yang hanya boleh memilih di antara 256 nilai tidak bisa setepat float32, jadi setiap bobot dan setiap nilai yang mengalir di dalam model ikut dibulatkan sedikit. Pertanyaan bab ini sebenarnya sederhana: seberapa besar kerusakan akibat pembulatan itu, di bagian model mana kerusakan itu terjadi, dan apakah model yang lebih kecil memang menjadi lebih cepat?

Pembaca Chapter 12 mungkin ingat bab itu ditutup dengan satu masalah yang belum selesai: satu baris `quantize_dynamic` justru membuat checkpoint GPT-2 *lebih besar* dan merusak loss-nya. Masalah yang sama akan muncul lagi di listing pertama bab ini, dan kali ini kita bongkar penyebabnya. Ternyata listing buku mengandaikan model yang sederhana: tanpa embedding, tanpa nilai ekstrem di aktivasinya, dan dengan lapisan `nn.Linear` biasa. GPT-2 tidak memenuhi ketiganya. Akibatnya banyak listing gagal saat dijalankan pada model sungguhan, atau berjalan tapi memberi hasil yang menyesatkan. Polanya akan selalu sama: listing dijalankan dulu apa adanya, kita lihat apa yang terjadi, lalu kita perbaiki.

Notebook ini berisi semua kode bab 13 dalam urutan buku. Di empat titik ada pengukuran tambahan yang tidak ada di buku, diberi judul **Under the hood**. Teksnya boleh dilewati kalau hanya ingin mengikuti materi buku, tapi selnya sebaiknya tetap dijalankan berurutan, karena Under the hood 4 memakai fungsi dan data kalibrasi dari Under the hood 2. Istilah di tabel berikut, seperti *scale*, *per-token*, kalibrasi, dan QAT, baru dijelaskan saat bagiannya tiba; untuk sekarang, kolom terakhir cukup dibaca sebagai gambaran apa yang akan kita temukan.

| Under the hood | Yang kita ukur | Temuan di run referensi |
|---|---|---|
| 1. Bobot atau aktivasi? | Kerusakan int8 di GPT-2, dipisah antara bobot dan aktivasi, dengan beberapa cara membagi scale | Bobot int8 dengan scale per baris hampir tidak merusak; kerusakan terbesar datang dari aktivasi, terutama satu dimensi input `lm_head` yang nilai absolutnya sampai 330 |
| 2. Kalibrasi PTQ statis | Tanpa kalibrasi, dengan token acak, dengan teks asli; dua jenis observer | Tanpa kalibrasi model rusak; token acak hampir sama baiknya dengan teks asli |
| 3. QAT di bawah 8 bit | QAT vs PTQ di model mini pada 4, 3, dan 2 bit, dengan kelompok kontrol | QAT baru jelas unggul di 2 bit |
| 4. Tabel format | fp32, fp16, bf16, autocast, dan beberapa varian int8: ukuran, loss, kecepatan | Tidak ada varian yang sekaligus memangkas sekitar 75% byte dan dua kali lebih cepat |

## Fondasi

Sebelum menyentuh quantization, kita siapkan dulu runtime-nya, lalu tiga hal yang dipakai sampai akhir bab: model yang akan dikompresi, penggaris untuk mengukur seberapa besar kerusakannya, dan alat untuk mengukur ukuran serta kecepatan. Karena penggaris dan alat ukurnya sama, setiap varian quantization nanti bisa dibandingkan satu sama lain dengan cara yang sama.

Notebook ini disiapkan untuk Google Colab dengan runtime CPU biasa. Quantization int8 `torch.ao` yang dipakai buku dirancang terutama untuk CPU, dan semua eksperimen di sini berjalan di CPU. GPU tidak dibutuhkan; kalaupun runtime-nya punya GPU, notebook ini tidak memakainya. Beberapa sel cukup berat, terutama Under the hood 2 dan 4, training model mini di bagian QAT, dan distilasi menjelang akhir bab. Masing-masing bisa memakan beberapa menit, tergantung CPU yang didapat dari Colab.

Output yang tersimpan di notebook ini berasal dari satu run referensi sebelumnya di mesin lain, bukan dari Colab: CPU Linux dengan 2 thread, torch 2.14, dan transformers 5.17. Beberapa sel kodenya diubah supaya notebook bisa dijalankan dengan *Run all* di Colab, dan sel-sel itu belum punya output: sel persiapan yang baru, sel fondasi, dan sel-sel yang kini diawali `%%boleh_gagal`, yang kebanyakan adalah listing buku. Angka yang dikutip di teks adalah angka run referensi. Kalau notebook ini dijalankan ulang, loss dan ukuran file biasanya dekat dengan angka di teks, walaupun bisa bergeser sedikit. Di eksperimen model mini, yang bergantung pada urutan bilangan acak, pergeserannya bisa lebih besar, jadi yang perlu dicocokkan di sana adalah polanya. Milidetik lain cerita: angka itu milik CPU yang mengukurnya, dan di CPU lain bisa berbeda jauh.

Seluruh notebook bisa dijalankan sekaligus lewat *Runtime > Run all*; listing buku yang gagal tidak akan menghentikannya. Kalau `HF_TOKEN` tersimpan di Colab Secrets, Colab bisa menampilkan dialog yang meminta izin agar notebook ini membaca secret itu, dan sel pertama menunggu sampai dialog itu dijawab. Mengizinkan atau menolak sama-sama aman, karena token ini tidak wajib.

Sel pertama menyiapkan runtime.

In [1]:
# Persiapan runtime: paket, token Hugging Face (opsional), dan satu magic untuk listing buku yang memang gagal.
# Dirancang untuk Google Colab (runtime CPU sudah cukup), tapi juga jalan di Jupyter lokal.
import os, sys, importlib, importlib.util, importlib.metadata, subprocess, platform, traceback, linecache

for var in ("HF_HUB_OFFLINE", "HF_DATASETS_OFFLINE", "TRANSFORMERS_OFFLINE"):   # bab ini mengunduh gpt2 dan wikitext dari Hub
    os.environ.pop(var, None)
_hub_constants = sys.modules.get("huggingface_hub.constants")
if _hub_constants is not None and _hub_constants.HF_HUB_OFFLINE:
    raise RuntimeError("Mode offline Hugging Face sudah terlanjur aktif di sesi ini: "
                       "pilih Runtime > Restart session, lalu jalankan ulang dari sel ini.")

def pip_install(*spec):
    env = dict(os.environ, PIP_ROOT_USER_ACTION="ignore", PIP_DISABLE_PIP_VERSION_CHECK="1")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *spec], env=env, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f"pip install {' '.join(spec)} gagal (exit {r.returncode}):\n{(r.stderr or r.stdout)[-2000:]}")

for pkg in ("transformers", "datasets"):            # Colab sudah membawa keduanya; di luar Colab dipasang kalau belum ada
    if importlib.util.find_spec(pkg) is None:
        pip_install(pkg)

import torch

def torchao_pin(torch_version):
    # torchao 0.17/0.18 hanya cocok untuk torch >= 2.11; rilis lama mengikuti torch-nya masing-masing
    major_minor = tuple(int(x) for x in torch_version.split("+")[0].split(".")[:2])
    if major_minor >= (2, 15): return None            # torch lebih baru dari tabel ini: pakai rilis torchao terbaru
    if major_minor >= (2, 11): return "0.18.0"
    if major_minor == (2, 10): return "0.16.0"
    if major_minor == (2, 9):  return "0.15.0"
    return "0.13.0"

pin = torchao_pin(torch.__version__)
try:
    installed = importlib.metadata.version("torchao")
except importlib.metadata.PackageNotFoundError:
    installed = None
if installed is None or (pin is not None and installed != pin):
    spec = "torchao" if pin is None else f"torchao=={pin}"
    print(f"memasang {spec} untuk torch {torch.__version__} ...")
    pip_install("--no-deps", spec)                    # --no-deps: torch bawaan Colab tidak disentuh
    importlib.invalidate_caches()
    if "torchao" in sys.modules:                      # versi lama sudah ada di memori: harus restart dulu
        raise RuntimeError(f"torchao {installed} sudah ter-import di sesi ini, sedangkan {spec} baru saja dipasang. "
                           "Pilih Runtime > Restart session, lalu jalankan ulang dari sel ini.")
import torchao
from torchao.quantization import quantize_, Int8DynamicActivationInt8WeightConfig, Int8WeightOnlyConfig

def setup_hf_token():
    # Token Hugging Face: opsional. gpt2 dan wikitext publik, jadi tanpa token pun tetap bisa diunduh.
    # Nilai token hanya hidup di dalam fungsi ini dan di environment proses; tidak dicetak, tidak ditulis ke file.
    try:
        from google.colab import userdata             # hanya ada di Colab
    except ImportError:
        return "dipakai, dari environment variable HF_TOKEN" if os.environ.get("HF_TOKEN") else "tidak dipakai (unduhan anonim)"
    try:
        token = (userdata.get("HF_TOKEN") or "").strip()
    except Exception as e:                            # SecretNotFoundError, NotebookAccessError, TimeoutException
        return f"tidak dipakai ({type(e).__name__})"
    if not token:
        return "tidak dipakai (secret kosong)"
    if os.environ.get("HF_HUB_DISABLE_IMPLICIT_TOKEN") == "1":
        return "pernah ditolak di sesi ini; Runtime > Restart session supaya token baru dipakai"
    import huggingface_hub
    try:
        huggingface_hub.whoami(token=token)           # cek token masih valid; info akunnya tidak dicetak
    except Exception as e:
        # token tidak bisa dipakai: jangan kirim token apa pun (Hub juga membaca Colab Secrets sendiri)
        os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
        huggingface_hub.constants.HF_HUB_DISABLE_IMPLICIT_TOKEN = True
        return f"tidak bisa dipakai ({type(e).__name__}); unduhan anonim"
    os.environ["HF_TOKEN"] = token
    return "dipakai, dari Colab Secrets (nilainya tidak ditampilkan)"

hf_token_status = setup_hf_token()

# %%boleh_gagal: menjalankan isi sel apa adanya di namespace notebook. Kalau sel itu gagal, traceback-nya
# dicetak, tapi sel tidak dihitung error, jadi "Run all" tidak berhenti di listing buku yang memang rusak.
# %%boleh_gagal --fresh: sama, tapi di namespace kosong, seperti listing yang dijalankan di kernel baru.
from IPython import get_ipython
from IPython.core.magic import register_cell_magic

_listing_counter = [0]

@register_cell_magic
def boleh_gagal(line, cell):
    _listing_counter[0] += 1
    fname = f"<listing-{_listing_counter[0]}>"
    linecache.cache[fname] = (len(cell), None, cell.splitlines(True), fname)   # supaya traceback bisa menampilkan barisnya
    namespace = {"__name__": "__main__"} if "--fresh" in line.split() else get_ipython().user_ns
    try:
        exec(compile(cell, fname, "exec"), namespace)
    except Exception as e:
        te = traceback.TracebackException.from_exception(e)
        te.stack = traceback.StackSummary.from_list(list(te.stack)[1:])       # buang frame milik magic ini sendiri
        print("".join(te.format()) + f"[sel ini berhenti di {type(e).__name__}; notebook tetap lanjut ke sel berikutnya]",
              file=sys.stderr)

def cpu_features():
    try:
        with open("/proc/cpuinfo") as f:
            lines = f.read().splitlines()
    except OSError:
        return "tidak terbaca di sistem ini", ""
    model = next((l.split(":", 1)[1].strip() for l in lines if l.startswith("model name")), "?")
    flags = next((l.split(":", 1)[1].split() for l in lines if l.startswith("flags")), [])
    return model, ", ".join(f"{name} {'ya' if name in flags else 'tidak'}"
                            for name in ("avx2", "avx512f", "avx512_vnni", "avx512_bf16", "amx_bf16", "amx_int8"))

import transformers, datasets
cpu_model, cpu_flags = cpu_features()
ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9 if hasattr(os, "sysconf") else float("nan")
print(f"python {platform.python_version()} | torch {torch.__version__} | transformers {transformers.__version__} "
      f"| datasets {datasets.__version__} | torchao {torchao.__version__}")
print(f"CPU: {cpu_model} | {os.cpu_count()} core logis, torch threads {torch.get_num_threads()}, RAM {ram_gb:.1f} GB "
      f"| GPU terdeteksi: {torch.cuda.is_available()} (bab ini tetap berjalan di CPU)")
print(f"fitur CPU: {cpu_flags} | kernel CPU torch: {torch.backends.cpu.get_cpu_capability()}")
print(f"engine quantized: {torch.backends.quantized.engine} | tersedia: {torch.backends.quantized.supported_engines}")
print(f"HF_TOKEN: {hf_token_status}")

memasang torchao==0.18.0 untuk torch 2.11.0+cu130 ...
python 3.13.15 | torch 2.11.0+cu130 | transformers 5.18.0 | datasets 4.8.5 | torchao 0.18.0
CPU: Intel(R) Xeon(R) CPU @ 2.00GHz | 2 core logis, torch threads 1, RAM 13.6 GB | GPU terdeteksi: True (bab ini tetap berjalan di CPU)
fitur CPU: avx2 ya, avx512f ya, avx512_vnni tidak, avx512_bf16 tidak, amx_bf16 tidak, amx_int8 tidak | kernel CPU torch: AVX512
engine quantized: x86 | tersedia: ['qnnpack', 'onednn', 'x86', 'fbgemm']
HF_TOKEN: dipakai, dari Colab Secrets (nilainya tidak ditampilkan)


Sel tadi belum menyentuh model. Ia hanya membuka jalan untuk sisa bab. Pengaturan mode offline Hugging Face yang mungkin tertinggal dibersihkan supaya GPT-2 dan wikitext-2 bisa diunduh. Lalu `torchao`, pustaka yang ditunjuk PyTorch sendiri sebagai pengganti API quantization lama, dipasang dalam versi yang cocok dengan torch di runtime ini, tanpa mengganti torch bawaan Colab. Kalau stderr-nya memuat beberapa baris `Failed to load ... .so`, itu hanya berarti kernel khusus GPU dari torchao tidak dimuat; jalur CPU yang kita pakai tidak terpengaruh. Kalau ada `HF_TOKEN` di Colab Secrets dan Hub menerimanya, token itu dipakai, dan nilainya tidak pernah dicetak atau ditulis ke file. Token ini tidak wajib, karena kedua bahan tadi publik; baris `HF_TOKEN` di akhir output memberi tahu apakah token dipakai.

Kalau sel itu berhenti dengan error yang meminta *Runtime > Restart session*, artinya mode offline Hugging Face atau torchao versi lain sudah terlanjur dimuat di sesi ini, misalnya karena notebook pernah dijalankan sebagian. Restart, lalu jalankan lagi mulai dari sel persiapan. Restart juga dibutuhkan sesudah memperbaiki secret yang ditolak: begitu pemeriksaan token gagal, token tidak dipakai lagi sampai sesi di-restart, dan baris `HF_TOKEN` akan mengingatkannya.

Sel persiapan juga mendefinisikan `%%boleh_gagal`. Bab ini sengaja menjalankan listing buku apa adanya, termasuk yang rusak, supaya errornya terlihat. Masalahnya, *Run all* berhenti di sel pertama yang error. *Magic* ini, yaitu baris berawalan `%%` di baris pertama sel, menjalankan isi sel persis seperti tertulis. Kalau gagal, traceback-nya dicetak (selnya tampil sebagai `<listing-1>`, `<listing-2>`, dan seterusnya, dengan nomor baris dihitung mulai sesudah baris magic), diikuti baris `[sel ini berhenti di ...; notebook tetap lanjut ke sel berikutnya]`, lalu notebook lanjut. Hanya sel yang memang diharapkan gagal yang memakainya. Jadi error di sel lain adalah masalah sungguhan, dan pemeriksaannya dimulai dari baris versi di output sel persiapan. Beberapa sel juga menghapus model yang sudah tidak dipakai, jadi jalankan notebook ini berurutan dari atas.

Baris `fitur CPU` di output sel persiapan baru kita pakai di bagian hardware, dan baris `engine quantized` sebentar lagi.

Sekarang fondasinya. Sel berikut memuat GPT-2 124M pretrained beserta tokenizer-nya, mengambil teks wikitext-2, lalu mendefinisikan tiga alat ukur yang dipakai sepanjang bab:

- `eval_gpt2` adalah penggaris kita: loss validasi pada 8 potongan teks wikitext-2, masing-masing 128 token. Arti angkanya kita bahas begitu angka pertama keluar.
- `nbytes` mengukur ukuran checkpoint, yaitu jumlah byte `state_dict` model saat disimpan ke disk. Ini ukuran file, bukan pemakaian memori GPU.
- `latency_ms` mengukur kecepatan: median waktu 7 forward pass pada batch 2 x 128 token, sesudah 2 forward pemanasan.

Sel yang sama juga mendefinisikan `conv1d_to_linear`, operasi kecil yang ternyata menjadi kunci bab ini. GPT-2 versi Hugging Face tidak memakai `nn.Linear` untuk proyeksi di dalam blok transformer-nya, tapi kelas bernama `Conv1D`. Secara matematis `Conv1D` sama dengan lapisan linear; bedanya hanya bobotnya disimpan dengan bentuk terbalik, (input, output) alih-alih (output, input). API quantization yang kita pakai di bab ini mencari `nn.Linear` dan tidak mengenali `Conv1D`. `conv1d_to_linear` mengganti setiap `Conv1D` dengan `nn.Linear` yang bobotnya ditranspos, sehingga hasil perhitungannya tetap sama.

Di bagian import-nya, sel fondasi juga mengikat `quant`, `quantize_dynamic`, dan `autocast` versi buku. Listing buku meng-import nama-nama ini sendiri, tapi sebagian listing itu gagal di tengah jalan. Dengan meng-import-nya di sini, sel-sel perbaikan tidak bergantung pada listing yang gagal.

In [2]:
import os
import sys, io, time, math, copy, gc, collections, warnings, zlib
import torch, numpy as np
import torch.nn as nn
import torch.nn.functional as F
import transformers, datasets
from transformers import GPT2LMHeadModel, GPT2Tokenizer, GPT2Config, AutoTokenizer
from transformers.pytorch_utils import Conv1D
from datasets import load_dataset
import torch.quantization as quant                 # nama-nama ini juga diimpor oleh listing buku yang gagal di tengah jalan;
from torch.quantization import quantize_dynamic   # diikat di sini supaya sel perbaikan tidak bergantung pada listing itu
try:
    from torch.cuda.amp import autocast           # autocast versi buku, dipakai lagi di bagian mixed precision
except ImportError:                               # kalau suatu saat jalur lama ini dihapus dari torch
    import functools; autocast = functools.partial(torch.amp.autocast, "cuda")

transformers.logging.set_verbosity_error()
warnings.filterwarnings("ignore", category=DeprecationWarning)            # deprecation torch.ao ditampilkan eksplisit di sel berikut
warnings.filterwarnings("ignore", message=".*reduce_range will be deprecated.*")
warnings.filterwarnings("ignore", message=".*quantized tensor creation functions.*")
torch.manual_seed(42)
print("python:", sys.version.split()[0], "| torch:", torch.__version__, "| transformers:", transformers.__version__,
      "| datasets:", datasets.__version__, "| threads:", torch.get_num_threads())

gpt2 = GPT2LMHeadModel.from_pretrained("gpt2")
gpt2_tok = GPT2Tokenizer.from_pretrained("gpt2"); gpt2_tok.pad_token = gpt2_tok.eos_token

wiki_val = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="validation")
wiki_train = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
def pack(split, n_chars, seq):
    text = ""
    for t in split["text"]:
        if t.strip(): text += t
        if len(text) >= n_chars: break
    ids = gpt2_tok(text, return_tensors="pt")["input_ids"][0]
    return ids[:len(ids) // seq * seq].view(-1, seq)
ft_val = pack(wiki_val, 16_000, 128)[:8]
ft_train = pack(wiki_train, 60_000, 128)

def eval_gpt2(m, data=ft_val):
    m.eval()
    with torch.no_grad():
        return float(np.mean([m(input_ids=data[i*2:(i+1)*2], labels=data[i*2:(i+1)*2]).loss.item()
                              for i in range(len(data)//2)]))

def nbytes(mm):
    torch.save(mm.state_dict(), "_size_probe.pt")
    n = os.path.getsize("_size_probe.pt"); os.remove("_size_probe.pt")
    return n

def latency_ms(m, x=None, reps=7):
    x = ft_val[:2] if x is None else x
    m.eval()
    with torch.inference_mode():
        for _ in range(2): m(input_ids=x)
        ts = []
        for _ in range(reps):
            t0 = time.perf_counter(); m(input_ids=x); ts.append(time.perf_counter() - t0)
    return 1000 * float(np.median(ts))

def conv1d_to_linear(model):
    n = 0
    for _, mod in list(model.named_modules()):
        for cname, child in list(mod.named_children()):
            if isinstance(child, Conv1D):
                lin = nn.Linear(child.weight.shape[0], child.weight.shape[1])
                lin.weight = nn.Parameter(child.weight.detach().t().contiguous())   # Conv1D (in, out) -> Linear (out, in)
                lin.bias = nn.Parameter(child.bias.detach().clone())
                setattr(mod, cname, lin); n += 1
    return n

base_val = eval_gpt2(gpt2)
fp32_mb = nbytes(gpt2) / 1e6
census_lin = [n for n, mod in gpt2.named_modules() if isinstance(mod, nn.Linear)]
census_conv = sum(1 for _, mod in gpt2.named_modules() if isinstance(mod, Conv1D))
print(f"val loss baseline: {base_val:.3f} | checkpoint fp32: {fp32_mb:.1f} MB | "
      f"tied lm_head<->wte: {gpt2.lm_head.weight.data_ptr() == gpt2.transformer.wte.weight.data_ptr()}")
print(f"sensus GPT-2: nn.Linear = {len(census_lin)} ({census_lin[0]}) | Conv1D = {census_conv}")


python: 3.13.15 | torch: 2.11.0+cu130 | transformers: 5.18.0 | datasets: 4.8.5 | threads: 1


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

...

wikitext-2-raw-v1/validation-00000-of-00(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

val loss baseline: 3.762 | checkpoint fp32: 497.8 MB | tied lm_head<->wte: True
sensus GPT-2: nn.Linear = 1 (lm_head) | Conv1D = 48


Sel tadi mencetak versi paket lalu dua baris angka. Di run referensi isinya: loss validasi 3.762, checkpoint fp32 497.8 MB, `lm_head` dan `wte` terikat (*tied*: `True`), dan sensus lapisan berisi satu `nn.Linear` (`lm_head`) serta 48 `Conv1D`.

Angka pertama akan menjadi penggaris untuk semua model di bab ini, jadi mari kita pahami baik-baik sejak awal. Loss 3.762 adalah rata-rata *cross-entropy* dari 1,016 tebakan token berikutnya: ada 8 potongan teks validasi, masing-masing 128 token, dan di setiap potongan model menebak 127 token sesudah token pertama. Untuk setiap tebakan, kita ambil peluang yang diberikan model kepada token yang benar, lalu hitung minus logaritma naturalnya; rata-ratanya itulah loss. Karena memakai logaritma natural, satuannya disebut *nats*. Cara membacanya: `e^3.762` kira-kira 43. Artinya, secara rata-rata model seyakin orang yang harus memilih di antara sekitar 43 kemungkinan yang sama besar. Angka `e^loss` ini dikenal sebagai *perplexity*. Sebagai pembanding, model yang menebak asal di antara 50,257 token GPT-2 akan mendapat loss sekitar 10.8.

Yang paling sering kita baca nanti adalah *selisih* loss terhadap 3.762. Selisih +0.01 nats berarti peluang rata-rata (geometrik) untuk token yang benar turun sekitar 1%; +0.1 nats berarti turun sekitar 10%; dan +1 nats berarti peluang itu tinggal kira-kira sepertiganya, karena dibagi `e`, sekitar 2.7. Jadi +0.005 boleh dianggap tidak terasa, sedangkan +1 adalah kerusakan besar.

Angka kedua bisa dihitung ulang: 124.4 juta parameter dikali 4 byte adalah 497.8 MB. Satu matriks layak diingat sejak sekarang. Embedding token `wte` berukuran 50,257 x 768, atau 38.6 juta parameter, hampir sepertiga seluruh model. GPT-2 memakainya dua kali: sebagai tabel embedding di awal, dan sebagai `lm_head`, lapisan terakhir yang mengubah vektor 768 dimensi menjadi skor untuk setiap token di kosakata. Itulah arti *tied*: dua peran, satu matriks, disimpan sekali.

Sensus lapisan, yang dicetak di baris terakhir sel itu, menjelaskan banyak hal yang akan terjadi sebentar lagi. Dari sudut pandang PyTorch, GPT-2 hanya punya satu `nn.Linear`, yaitu `lm_head`. Ke-48 proyeksi di dalam blok, empat per blok di 12 blok, semuanya `Conv1D`. Resep quantization yang mencari `nn.Linear` akan melewatkan hampir seluruh model.

Progress bar unduhan atau peringatan token dari Hugging Face yang mungkin muncul di stderr Colab tidak mengubah angka-angka ini.

Sebelum listing pertama, ada satu pertanyaan yang tidak diajukan buku: apa kata torch sendiri tentang API quantization yang dipakai bab ini?

In [3]:
# Status API yang dipakai buku, ditanyakan langsung ke torch yang terpasang
import textwrap
from torch.ao.quantization import quantize_dynamic as _qd
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    _qd(nn.Sequential(nn.Linear(8, 8)), {nn.Linear}, dtype=torch.qint8)
print(f"torch {torch.__version__} | quantized engine: {torch.backends.quantized.engine} "
      f"| tersedia: {torch.backends.quantized.supported_engines}")
seen = set()
for w in caught:
    msg = " ".join(str(w.message).split())
    if msg in seen: continue
    seen.add(msg)
    print(f"\n[{w.category.__name__}]")
    print(textwrap.fill(msg, 110))


torch 2.11.0+cu130 | quantized engine: x86 | tersedia: ['qnnpack', 'onednn', 'x86', 'fbgemm']

[DeprecationWarning]
torch.ao.quantization is deprecated and will be removed in 2.10. For migrations of users: 1. Eager mode
quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use
torchao eager mode quantize_ API instead 2. FX graph mode quantization
(torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to
use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 3. pt2e quantization has been migrated
to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) see
https://github.com/pytorch/ao/issues/2259 for more details


Sel tadi memanggil `quantize_dynamic` dari `torch.ao.quantization` pada model kecil, lalu mencetak setiap peringatan yang muncul. Output yang tersimpan di sel itu berasal dari run referensi dengan torch 2.14, dan jawabannya jelas. Peringatan pertama, `DeprecationWarning`, menyatakan seluruh `torch.ao.quantization` sudah *deprecated* (ditandai usang) dan dijadwalkan dihapus di versi 2.10. Itu mencakup semua yang dipakai buku: `quantize_dynamic`, `prepare`, `convert`, dan `prepare_qat`. Pesannya juga menyebut ke mana harus pindah: untuk *eager mode quantization*, gaya yang dipakai buku, penggantinya adalah API `quantize_` di torchao.

Torch 2.14 sudah melewati versi 2.10, tapi API lamanya masih bisa dipanggil. Jadi listing buku tetap bisa dijalankan, hanya arah migrasinya sudah pasti. Kalau baris pertama output sel tadi menunjukkan torch 2.10 atau lebih baru, tenggat itu juga sudah lewat di runtime yang sedang dipakai; selama sel-sel berikutnya masih bisa memanggil `torch.ao.quantization`, API itu belum benar-benar dihapus di versi tersebut. Teks peringatannya juga bisa sedikit berbeda dari versi ke versi. Di torch 2.14 ada peringatan kedua, sebuah `UserWarning`, yang bahkan menandai fungsi-fungsi pembuat tensor ter-quantisasi, yaitu yang menghasilkan tensor ber-dtype `torch.quint8`, `torch.qint8`, dan `torch.qint32`, sebagai usang; di versi torch lain peringatan ini mungkin belum muncul.

Baris pertama yang sama juga menyebut *quantized engine*, yaitu kumpulan kernel yang dipakai PyTorch untuk menghitung int8 di CPU. Untuk CPU Intel dan AMD, engine bawaannya `x86`, yang menggabungkan kernel FBGEMM dan oneDNN. Listing buku masih memakai `'fbgemm'`, engine yang menjadi bawaan sebelum `x86` dan masih ada di daftar engine yang tersedia; keduanya akan kita temui lagi.

Rencana bab ini mengikuti pesan tadi. Listing buku dijalankan dengan API lama, persis seperti dicetak, dan ketika API lama tidak sanggup, torchao menjadi jalur perbaikannya.

## Understanding the basics

Quantization berarti menyimpan dan menghitung angka dengan presisi yang lebih rendah: dari float32 ke float16 atau bfloat16 (2 byte), atau ke int8 (1 byte). Ke float16 atau bfloat16, cukup membulatkan angka ke format yang lebih pendek. Int8 butuh langkah tambahan, karena hanya punya 256 bilangan bulat, dari -128 sampai 127. Setiap angka asli harus dipetakan ke salah satu bilangan itu, dan pemetaannya ditentukan dua parameter. *Scale* adalah jarak antara dua bilangan int8 yang bersebelahan bila diterjemahkan kembali ke float. *Zero point* adalah bilangan bulat yang mewakili angka 0.

Supaya lebih jelas, kita ambil contoh kecil. Misalkan semua bobot sebuah matriks berada di antara -0.5 dan 0.5. Dengan zero point 0, nilai 0.5 dipetakan ke 127, jadi scale-nya 0.5 / 127 = 0.003937. Bobot 0.1234 dibagi scale menjadi 31.3, dibulatkan menjadi 31, lalu disimpan sebagai satu byte. Saat dipakai, ia dibaca kembali sebagai 31 x 0.003937 = 0.1220. Galatnya 0.0014, dan tidak pernah lebih dari setengah scale.

Sekarang bayangkan matriks yang sama punya satu bobot bernilai 5.0. Supaya bobot itu terjangkau, scale-nya harus naik sepuluh kali lipat menjadi 5 / 127 = 0.03937. Bobot 0.1234 kini dibulatkan ke 3 dan dibaca kembali sebagai 3 x 0.03937 = 0.1181, dan setiap bobot yang lebih kecil dari sekitar 0.02 dibulatkan menjadi nol. **Rentang yang harus dijangkau satu scale menentukan seberapa kasar pembulatan untuk semua nilai lainnya.** Contoh bobot 5.0 ini akan kita pakai lagi beberapa kali, karena prinsipnya muncul di hampir setiap eksperimen bab ini.

Pemetaan dengan zero point 0 seperti tadi disebut *simetris*, dan cocok untuk nilai yang tersebar di sekitar nol, seperti bobot. Kalau nilainya tidak berpusat di nol, misalnya semuanya positif sesudah ReLU, zero point bisa digeser supaya bilangan-bilangan bulat yang tersedia hanya dipakai untuk rentang yang benar-benar terisi. Pemetaan seperti ini disebut *asimetris* atau *affine*. Zero point juga berperan kalau bilangan bulatnya tak bertanda, seperti `quint8` (0 sampai 255) yang dipakai `torch.ao` untuk aktivasi. Untuk nilai yang tersebar di sekitar nol, angka 0 harus diletakkan kira-kira di tengah deretan bilangan yang dipakai. Di bab ini aktivasi `torch.ao` sering hanya memakai separuh deretan itu, 0 sampai 127, karena alasan yang akan kita lihat nanti, dan itulah sebabnya kita akan bertemu zero point seperti 64 atau 65.

Di sebuah lapisan ada dua hal yang bisa di-quantize: bobotnya, yang tetap dan sudah diketahui sebelum model dipakai, dan *aktivasi*, yaitu input yang mengalir masuk ke lapisan itu dan berubah untuk setiap teks. Buku membedakan dua cara menangani aktivasi. Pada *dynamic quantization*, scale aktivasi dihitung ulang saat inferensi, dari rentang nilai yang sedang lewat. Pada *static quantization*, scale itu dihitung sekali dari data contoh, lalu dibekukan.

Listing dynamic quantization pertama di buku ditutup dengan janji *"This immediately reduces memory requirements and increases inference speed."* Kita jalankan apa adanya. Inilah sel pertama yang memakai `%%boleh_gagal`.

In [4]:
%%boleh_gagal
import torch
from torch.quantization import quantize_dynamic
# Assume 'model' is a pre-trained LLM (e.g., transformer-based model)
model = ...
# Apply dynamic quantization on linear layers for INT8 precision
quantized_model = quantize_dynamic(
    model, {torch.nn.Linear}, dtype=torch.qint8
)
# Check size reduction
print(f"Original model size: {torch.cuda.memory_allocated()} bytes")
print(f"Quantized model size: {torch.cuda.memory_allocated()} bytes")


Traceback (most recent call last):
  File "<listing-1>", line 6, in <module>
    quantized_model = quantize_dynamic(
        model, {torch.nn.Linear}, dtype=torch.qint8
    )
  File "/usr/lib/python3.13/warnings.py", line 637, in wrapper
    return arg(*args, **kwargs)
  File "/usr/local/lib/python3.13/dist-packages/torch/ao/quantization/quantize.py", line 568, in quantize_dynamic
    model.eval()
    ^^^^^^^^^^
AttributeError: 'ellipsis' object has no attribute 'eval'
[sel ini berhenti di AttributeError; notebook tetap lanjut ke sel berikutnya]


Begitu dijalankan, sel ini berhenti dengan `AttributeError: 'ellipsis' object has no attribute 'eval'`. Pesan ini datang dari kodenya sendiri, bukan dari mesinnya, jadi run referensi pun berhenti di tempat yang sama.

Penyebabnya baris `model = ...`. Di buku, tiga titik itu dimaksudkan sebagai "isi dengan model yang ingin di-quantize". Bagi Python, `...` adalah objek sungguhan bernama `Ellipsis`, jadi baris itu berjalan tanpa protes. Masalahnya baru muncul di dalam `quantize_dynamic`, yang salah satu langkah pertamanya memanggil `model.eval()`, method yang tidak dimiliki `Ellipsis`. Placeholder yang ikut dieksekusi seperti ini akan kita temui beberapa kali lagi.

Perbaikannya: isi placeholder dengan GPT-2 yang sudah kita muat, dan jalankan sisa listing apa adanya, termasuk dua baris yang dimaksudkan buku untuk mengukur ukuran model. Sesudahnya kita ukur ulang dengan alat ukur kita sendiri: berapa lapisan yang benar-benar ter-quantisasi, ukuran checkpoint, loss, dan kecepatan.

In [5]:
model = gpt2                                              # perbaikan #1: placeholder diisi LLM sungguhan
quantized_model = quantize_dynamic(
    model, {torch.nn.Linear}, dtype=torch.qint8
)
# Check size reduction -- persis buku
print(f"Original model size: {torch.cuda.memory_allocated()} bytes")
print(f"Quantized model size: {torch.cuda.memory_allocated()} bytes")

print("--- diukur ulang dengan penggaris yang benar ---")
qnames = [n for n, m in quantized_model.named_modules()
          if type(m).__module__.startswith("torch.ao.nn.quantized.dynamic")]
loss_book_dyn = eval_gpt2(quantized_model)
print(f"modul ter-quantisasi    : {len(qnames)} dari {len(census_lin) + census_conv} proyeksi -> {qnames}")
print(f"checkpoint              : {fp32_mb:.1f} MB -> {nbytes(quantized_model)/1e6:.1f} MB")
print(f"val loss                : {base_val:.3f} -> {loss_book_dyn:.3f}")
print(f"latensi forward bs2x128 : {latency_ms(model):.0f} ms -> {latency_ms(quantized_model):.0f} ms")
del quantized_model; gc.collect();                        # ";" agar Colab tidak menampilkan hasil gc.collect()


Original model size: 0 bytes
Quantized model size: 0 bytes
--- diukur ulang dengan penggaris yang benar ---
modul ter-quantisasi    : 1 dari 49 proyeksi -> ['lm_head']
checkpoint              : 497.8 MB -> 536.4 MB
val loss                : 3.762 -> 8.995
latensi forward bs2x128 : 1318 ms -> 1178 ms


Dua baris pengukuran buku mencetak `0` dan `0`. `torch.cuda.memory_allocated()` membaca memori yang sedang terpakai di GPU, padahal model kita ada di CPU, dan dynamic quantization `torch.ao` memang dikerjakan di CPU. Lagi pula kedua baris itu baru dipanggil *sesudah* quantization, jadi di GPU pun keduanya akan mencetak angka yang sama.

Alat ukur kita, di run referensi, bercerita lain. Dari 49 proyeksi di GPT-2, hanya satu yang ter-quantisasi, yaitu `lm_head`, persis seperti yang diramalkan sensus tadi. Akibatnya checkpoint justru *membesar* dari 497.8 ke 536.4 MB. Selisihnya 38.6 MB, tepat sebesar matriks 50,257 x 768 dalam int8. `lm_head` mendapat salinan int8 miliknya sendiri, sementara `wte` tetap harus disimpan dalam float32 karena masih dipakai sebagai tabel embedding. Ikatan *tied* putus, dan matriks terbesar di model kini tersimpan dua kali.

Loss-nya lebih parah lagi: dari 3.762 ke 8.995. Dengan cara membaca tadi, model berubah dari memilih di antara sekitar 43 kemungkinan menjadi sekitar 8,000, mendekati tebakan asal (10.8). Dan kecepatannya nyaris tidak berubah: 493 ke 479 ms.

Kecepatan yang nyaris tidak bergeser ini sebenarnya bisa diperkirakan sebelum diukur. Pekerjaan utama lapisan linear adalah *MAC* (*multiply-accumulate*): satu perkalian ditambah satu penjumlahan untuk setiap pasangan input dan bobot. Untuk satu token, `lm_head` mengerjakan 768 x 50,257, sekitar 38.6 juta MAC. Empat proyeksi di setiap blok berukuran 768 x 2,304, 768 x 768, 768 x 3,072, dan 3,072 x 768, totalnya sekitar 7.1 juta MAC per blok, atau 84.9 juta untuk 12 blok. Jadi `lm_head` hanya memikul sekitar 31% pekerjaan. Kalaupun lapisan itu menjadi gratis, forward paling banter sekitar 1.45 kali lebih cepat (1 / 0.69). Hitungan ini mengandaikan waktu sebanding dengan jumlah MAC, dan mengabaikan LayerNorm, GELU, softmax di attention, serta overhead lain.

Jadi di model sungguhan janji buku berbalik: checkpoint lebih besar, kecepatan hampir sama, dan akurasi runtuh. Sasaran yang seharusnya adalah ke-48 `Conv1D` di dalam blok, dan setelah `conv1d_to_linear`, `quantize_dynamic` akhirnya bisa melihat mereka.

Sel berikut membandingkan enam skema. Selain memilih sasaran (blok dan `lm_head`, blok saja, atau `lm_head` saja), kita memilih cara menghitung scale bobot. *Per-tensor* berarti satu scale untuk seluruh matriks. *Per-channel* berarti satu scale untuk setiap baris, yaitu setiap neuron output. Ingat contoh bobot 5.0 tadi: dengan per-tensor, satu bobot besar membuat seluruh matriks dibulatkan kasar; dengan per-channel, hanya baris yang memuat bobot besar itu yang terkena.

In [6]:
from torch.ao.quantization import default_dynamic_qconfig, per_channel_dynamic_qconfig
g = copy.deepcopy(gpt2)
n_conv = conv1d_to_linear(g)
print(f"conv1d_to_linear: {n_conv} Conv1D -> nn.Linear | val {eval_gpt2(g):.4f} (asli {base_val:.4f}) | "
      f"tied tetap: {g.lm_head.weight.data_ptr() == g.transformer.wte.weight.data_ptr()} | fp32 {latency_ms(g):.0f} ms\n")
schemes = [
    ("per-tensor, 48 blok + lm_head",       {nn.Linear: default_dynamic_qconfig}),
    ("per-channel, 48 blok + lm_head",      {nn.Linear: per_channel_dynamic_qconfig}),
    ("per-tensor, 48 blok (lm_head fp32)",  {"transformer.h": default_dynamic_qconfig}),
    ("per-channel, 48 blok (lm_head fp32)", {"transformer.h": per_channel_dynamic_qconfig}),
    ("per-tensor, lm_head saja",            {"lm_head": default_dynamic_qconfig}),
    ("per-channel, lm_head saja",           {"lm_head": per_channel_dynamic_qconfig}),
]
dyn_rows = {}
print(f"{'skema quantize_dynamic (qint8)':<38} {'Linear int8':>11} {'val loss':>9} {'MB':>7} {'ms':>5}")
for label, spec in schemes:
    q = quantize_dynamic(g, spec, dtype=torch.qint8)
    nq = sum(1 for m in q.modules() if type(m).__module__.startswith("torch.ao.nn.quantized.dynamic"))
    dyn_rows[label] = (eval_gpt2(q), nbytes(q) / 1e6, latency_ms(q))
    print(f"{label:<38} {nq:>11} {dyn_rows[label][0]:>9.3f} {dyn_rows[label][1]:>7.1f} {dyn_rows[label][2]:>5.0f}")
    del q; gc.collect()


conv1d_to_linear: 48 Conv1D -> nn.Linear | val 3.7621 (asli 3.7621) | tied tetap: True | fp32 1134 ms

skema quantize_dynamic (qint8)         Linear int8  val loss      MB    ms
per-tensor, 48 blok + lm_head                   49     9.602   281.6   954
per-channel, 48 blok + lm_head                  49     9.187   283.8   543
per-tensor, 48 blok (lm_head fp32)              48     4.453   243.0   579
per-channel, 48 blok (lm_head fp32)             48     4.071   244.4   592
per-tensor, lm_head saja                         1     8.995   536.4   667
per-channel, lm_head saja                        1     8.856   537.2   658


Penggantian itu tidak mengubah perilaku model. Ke-48 `Conv1D` menjadi `nn.Linear`, loss-nya sama sampai digit keempat (3.7621), dan ikatan *tied* tetap utuh. Salinan GPT-2 versi `nn.Linear` ini disimpan dengan nama `g`, dan dialah yang dipakai di hampir semua eksperimen sampai bagian distilasi.

Sekarang tabel enam skema di bawah baris itu. Lihat dulu dua baris yang hanya menyasar 48 proyeksi blok, dengan `lm_head` tetap float32. Di run referensi checkpoint-nya turun ke sekitar 244 MB, kira-kira separuh. Kenapa bukan seperempat? Karena embedding `wte` masih float32, dan matriks 50,257 x 768 itu saja sudah 154 MB, hampir dua pertiga checkpoint yang tersisa. Kita akan kembali ke embedding ini. Skema yang ikut meng-int8-kan `lm_head` justru lebih besar, dengan alasan yang sama seperti di sel sebelumnya: salinan int8 `lm_head` memutus ikatan *tied*.

Per-channel jelas lebih baik untuk bobot. Untuk 48 blok, kerusakan turun dari +0.70 nats (per-tensor) ke +0.32 nats (per-channel), dan harganya hanya 1.4 MB tambahan untuk menyimpan scale setiap baris. (Kolom ms kita tunda sampai Under the hood 4. Kalau ingin melirik, bandingkan hanya dengan angka float32 di baris pertama output yang sama, 378 ms, bukan dengan 493 ms di sel sebelumnya, yang mengukur versi `Conv1D` pada saat yang berbeda.)

Lalu ada yang aneh: setiap skema yang menyentuh `lm_head` berakhir di loss 8.9 sampai 9.6, apa pun cara menghitung scale bobotnya. Per-channel hampir tidak menolong di sana (8.995 ke 8.856). Kalau masalahnya ada di bobot, per-channel seharusnya memperbaikinya.

Petunjuk untuk keanehan `lm_head` ada di cara kerja `quantize_dynamic`. Fungsi itu meng-quantize dua hal sekaligus: bobot setiap lapisan, sekali saat konversi, dan input lapisan itu, yaitu aktivasinya, setiap kali forward, dengan satu scale untuk seluruh tensor input. Di engine `x86`, aktivasi itu bahkan hanya memakai 7 bit, 128 nilai alih-alih 256, karena pengaturan `reduce_range`. Pengaturan ini pengaman bawaan `torch.ao`: satu bit dikorbankan supaya penjumlahan hasil kali int8 tidak meluap (*overflow*) di instruksi yang dipakai sebagian CPU x86. Buku tidak pernah memisahkan kedua sumber kerusakan ini. Kita pisahkan sekarang.

### Under the hood 1: bobot atau aktivasi?

Alatnya disebut *fake quantization*: sebuah nilai di-quantize lalu langsung dikembalikan ke float. Nilai itu kini membawa galat pembulatan yang sama seperti versi int8-nya, tapi perhitungan selanjutnya tetap berjalan di float. Dengan cara ini kita bebas memilih apa yang dibulatkan, hanya bobot, hanya aktivasi, atau keduanya, dan di lapisan mana.

Sel berikut (di komentar kodenya disebut UTH-1) membuat simulator fake-quant yang mengikuti rumus `torch.ao`, lalu mencobanya di dua sasaran: `lm_head` saja, dan 48 blok saja. Untuk aktivasi ada satu pilihan baru, *per-token*: setiap token mendapat scale sendiri, dihitung dari nilai-nilai di vektor input token itu (768 angka untuk sebagian besar lapisan, 3,072 untuk input `mlp.c_proj`), alih-alih satu scale untuk seluruh tensor. (Di kodenya, scale bobot adalah nilai absolut terbesar dibagi 127.5, bukan 127 seperti contoh di awal bab, mengikuti observer bawaan `torch.ao`; bedanya hanya sekitar 0.4%.)

Sebelum dipercaya, simulatornya diuji dulu. Aktivasi per-tensor 7-bit, ditambah bobot yang di-quantize dengan cara yang sama seperti di sel sebelumnya, seharusnya mereproduksi angka `quantize_dynamic`. Kita cek di `lm_head` dengan bobot per-tensor, dan di 48 blok dengan bobot per-channel.

Di bagian akhir, sel ini juga memotret input beberapa lapisan pada dua potongan teks validasi: nilai absolut terbesar, mediannya, rasio keduanya, dan tiga dimensi yang nilainya paling besar. Nilai yang jauh lebih besar daripada sebagian besar nilai lain seperti ini disebut *outlier*.

In [7]:
# === UNDER THE HOOD (UTH-1): bobot atau aktivasi -- siapa yang membunuh int8 GPT-2? ===
gu = copy.deepcopy(g)
gu.lm_head.weight = nn.Parameter(gu.transformer.wte.weight.detach().clone())   # lepas tie: seperti quantize_dynamic, lm_head memegang salinan sendiri
W0 = {n: m.weight.detach().clone() for n, m in gu.named_modules() if isinstance(m, nn.Linear)}

def fq_w(W, per_channel, bits=8):                    # bobot: qint8 simetris, rumus observer default torch.ao
    qmax, qmin = 2 ** (bits - 1) - 1, -2 ** (bits - 1)
    amax = W.abs().amax(dim=1, keepdim=True) if per_channel else W.abs().max()
    scale = (amax / ((qmax - qmin) / 2)).clamp(min=1e-12)
    return (W / scale).round().clamp(qmin, qmax) * scale

def fq_a(x, mode, levels):                            # aktivasi: affine dinamis, per-tensor atau per-token
    mn, mx = (x.min(), x.max()) if mode == "tensor" else (x.amin(-1, keepdim=True), x.amax(-1, keepdim=True))
    mn, mx = mn.clamp(max=0.0), mx.clamp(min=0.0)
    scale = ((mx - mn) / levels).clamp(min=1e-12); zp = (-mn / scale).round()
    return ((x / scale + zp).round().clamp(0, levels) - zp) * scale

hooks = []
def set_fq(targets, w=None, a=None, levels=127):
    for h in hooks: h.remove()
    hooks.clear()
    for n, m in gu.named_modules():
        if isinstance(m, nn.Linear):
            sel = any(n.startswith(t) for t in targets)
            m.weight.data.copy_(fq_w(W0[n], w == "pc") if (sel and w) else W0[n])
            if sel and a:
                hooks.append(m.register_forward_pre_hook(lambda mod, inp, a=a, lv=levels: (fq_a(inp[0], a, lv),)))

arms = [("W per-tensor", dict(w="pt")),
        ("W per-channel", dict(w="pc")),
        ("A per-tensor 7-bit", dict(a="tensor", levels=127)),
        ("A per-tensor 8-bit", dict(a="tensor", levels=255)),
        ("A per-token 7-bit", dict(a="token", levels=127)),
        ("W pt + A per-tensor 7-bit (= resep torch)", dict(w="pt", a="tensor", levels=127)),
        ("W pc + A per-tensor 7-bit (= resep torch)", dict(w="pc", a="tensor", levels=127)),
        ("W pc + A per-token 7-bit", dict(w="pc", a="token", levels=127))]
print(f"{'fake-quant (sisanya fp32)':<44} {'lm_head saja':>12} {'48 blok':>9}")
uth1 = {}
for label, kw in arms:
    r = []
    for tg in (["lm_head"], ["transformer.h"]):
        set_fq(tg, **kw); r.append(eval_gpt2(gu))
    uth1[label] = r
    print(f"{label:<44} {r[0]:>12.4f} {r[1]:>9.4f}")
set_fq([])
print(f"\nvalidasi simulator vs quantize_dynamic asli -- lm_head pt: "
      f"{uth1['W pt + A per-tensor 7-bit (= resep torch)'][0]:.4f} vs {dyn_rows['per-tensor, lm_head saja'][0]:.4f} | "
      f"48 blok pc: {uth1['W pc + A per-tensor 7-bit (= resep torch)'][1]:.4f} vs {dyn_rows['per-channel, 48 blok (lm_head fp32)'][0]:.4f}")

stats = {}
def cap(name):
    def f(mod, inp):
        x = inp[0].detach().abs().flatten(0, -2)                     # (token, fitur)
        stats[name] = (x.max().item(), x.median().item(), x.amax(0).topk(3).indices.tolist(), x.amax(0).topk(3).values.tolist())
    return f
hs = [m.register_forward_pre_hook(cap(n)) for n, m in gu.named_modules() if isinstance(m, nn.Linear)]
with torch.no_grad(): gu(input_ids=ft_val[:2])
for h in hs: h.remove()
print(f"\n{'input ke layer':<29} {'max|x|':>8} {'median|x|':>10} {'rasio':>6}   3 dimensi outlier teratas")
for n in ["transformer.h.0.attn.c_attn", "transformer.h.0.mlp.c_fc", "transformer.h.0.mlp.c_proj",
          "transformer.h.5.attn.c_attn", "transformer.h.11.mlp.c_fc", "transformer.h.11.mlp.c_proj", "lm_head"]:
    mx, med, idx, vals = stats[n]
    print(f"{n:<29} {mx:>8.2f} {med:>10.3f} {mx/med:>6.0f}   {idx} = {[round(v, 1) for v in vals]}")
del gu, W0; gc.collect();                                 # ";" agar Colab tidak menampilkan hasil gc.collect()


fake-quant (sisanya fp32)                    lm_head saja   48 blok
W per-tensor                                       3.7690    3.9592
W per-channel                                      3.7772    3.7603
A per-tensor 7-bit                                 8.8203    4.0400
A per-tensor 8-bit                                 5.1647    3.8562
A per-token 7-bit                                  5.7486    3.7666
W pt + A per-tensor 7-bit (= resep torch)          8.9953    4.4380
W pc + A per-tensor 7-bit (= resep torch)          8.8563    4.0554
W pc + A per-token 7-bit                           5.7759    3.7814

validasi simulator vs quantize_dynamic asli -- lm_head pt: 8.9953 vs 8.9953 | 48 blok pc: 4.0554 vs 4.0712

input ke layer                  max|x|  median|x|  rasio   3 dimensi outlier teratas
transformer.h.0.attn.c_attn       0.80      0.073     11   [674, 709, 480] = [0.8, 0.8, 0.7]
transformer.h.0.mlp.c_fc          2.85      0.083     34   [674, 640, 55] = [2.9, 1.8, 1.6]
transform

Mulai dari pengujian simulatornya. Di `lm_head`, simulator dan `quantize_dynamic` sama sampai digit keempat (8.9953 dan 8.9953). Di 48 blok selisihnya sekitar 0.02 (4.0637 dan 4.0857). Sumber selisih kecil ini tidak kita telusuri; kemungkinan ada detail kernel int8, misalnya cara memilih scale atau zero point aktivasi, yang tidak ditiru persis oleh simulator. Untuk memisahkan efek bobot dari efek aktivasi, simulator yang semirip ini sudah cukup.

Hasilnya cukup tegas: **kerusakan int8 terbesar di GPT-2 datang dari aktivasi, bukan dari bobot.**

Bobot int8 sendirian hanya merusak sedikit. Di `lm_head` loss naik 0.007 dengan bobot per-tensor, dan 0.015 dengan per-channel; keduanya terlalu kecil untuk ditafsirkan lebih jauh. Di 48 blok, bobot per-tensor menambah sekitar 0.20, dan bobot per-channel menghapusnya sama sekali (3.7603, sedikit di bawah 3.762, yang berarti selisihnya hanya derau). Jadi bobot int8 hampir tidak merusak, asal scale-nya dihitung per baris.

Aktivasi per-tensor 7-bit sendirian sudah membawa `lm_head` ke 8.82, hampir seluruh kerusakan yang kita lihat di sel sebelumnya. Tabel anatomi di bagian bawah output menjelaskan kenapa. Input `lm_head` adalah keluaran `ln_f`, LayerNorm terakhir GPT-2, dan di sana ada satu dimensi, nomor 496, yang nilai absolutnya mencapai 330. Padahal median nilai absolut seluruh input `lm_head` hanya 0.31, jadi rasionya lebih dari seribu. Ini cerita bobot 5.0 di awal bab, hanya jauh lebih ekstrem, dan kali ini menimpa aktivasi. Satu scale untuk seluruh tensor, dengan 128 nilai, harus menjangkau 330, sehingga jarak antara dua nilai sedikitnya 330 / 127 = 2.6. Semua nilai yang lebih kecil dari setengah jarak itu, sekitar 1.3, dibulatkan menjadi nol. Karena mediannya 0.31, sedikitnya separuh input `lm_head`, dan kemungkinan jauh lebih banyak, hilang menjadi nol. Satu bit tambahan (8-bit, 256 nilai) hanya memperbaikinya ke 5.16, masih jauh dari 3.762.

Bagaimana dengan per-token? Untuk 48 blok, aktivasi per-token hampir gratis: 3.766, hanya +0.004, dan bila digabung dengan bobot per-channel 3.778 (+0.016). Tapi `lm_head` tetap rusak di 5.75. Kalau outlier-nya hanya muncul di segelintir token, scale per-token seharusnya menyelamatkan token-token lainnya. Hasil ini menunjukkan outlier-nya tidak sesempit itu; kemungkinan besar dimensi 496, 430, dan 36 bernilai besar di sebagian besar token, sehingga hampir setiap token tetap mendapat scale yang kasar. Tabel anatomi hanya mencatat nilai terbesar di antara 256 token, jadi bagian ini adalah dugaan yang cocok dengan angka, bukan hasil hitungan langsung.

Outlier juga ada di dalam blok, walaupun jauh lebih kecil. Di antara input blok yang kita intip, rasio terbesar ada di `mlp.c_proj`, proyeksi sesudah GELU: 93 di layer 0 dan 183 di layer 11. Dua angka ini akan berguna lagi di bagian mixed precision.

Diagnosisnya sekarang lengkap. Untuk 48 blok, resep yang tepat adalah bobot per-channel ditambah aktivasi per-token. Untuk `lm_head`, aktivasinya jangan di-quantize sama sekali: cukup bobotnya (*weight-only*), atau biarkan presisi penuh. Dynamic quantization `torch.ao` tidak menyediakan aktivasi per-token. Penggantinya, torchao, menyediakannya, dan inilah jalur migrasi yang ditunjuk pesan deprecation di awal bab.

In [8]:
# Jalur migrasi resmi (pesan deprecation di atas): torchao quantize_ -- aktivasi int8 PER-TOKEN + bobot int8 per-channel
import torchao   # dipasang oleh sel persiapan di awal notebook -- bukan resep buku; jalur yang ditunjuk torch sendiri
from torchao.quantization import quantize_, Int8DynamicActivationInt8WeightConfig, Int8WeightOnlyConfig

def ao_int8(model, embeddings=True):
    if embeddings:   # embedding: weight-only (tanpa kuantisasi aktivasi -- outlier ln_f di input lm_head tak disentuh)
        quantize_(model, Int8WeightOnlyConfig(), filter_fn=lambda m, fqn: fqn == "transformer.wte")
        model.lm_head.weight = model.transformer.wte.weight            # quantize_ memutus tie; pasang lagi
    quantize_(model, Int8DynamicActivationInt8WeightConfig(),
              filter_fn=lambda m, fqn: isinstance(m, nn.Linear) and fqn.startswith("transformer.h"))
    return model

print(f"torchao {torchao.__version__}")
ao_rows = {}
for label, emb in (("torchao W8A8 per-token (48 blok)", False), ("torchao W8A8 + embedding W8 (tied)", True)):
    q = ao_int8(copy.deepcopy(g), embeddings=emb)
    ao_rows[label] = (eval_gpt2(q), nbytes(q) / 1e6, latency_ms(q))
    v, mb, ms = ao_rows[label]
    print(f"{label:<36}: val {v:.3f} ({v - base_val:+.3f}) | {mb:.1f} MB ({mb / fp32_mb - 1:+.0%}) | {ms:.0f} ms "
          f"| tied: {q.lm_head.weight is q.transformer.wte.weight}")
    del q; gc.collect()


torchao 0.18.0
torchao W8A8 per-token (48 blok)    : val 3.767 (+0.005) | 243.5 MB (-51%) | 15902 ms | tied: True
torchao W8A8 + embedding W8 (tied)  : val 3.780 (+0.018) | 127.9 MB (-74%) | 15717 ms | tied: True


`Int8DynamicActivationInt8WeightConfig` dari torchao mengerjakan resep yang disarankan Under the hood 1: bobot int8 per-channel, dan aktivasi int8 per-token yang dihitung saat inferensi. *Granularitas*-nya, yaitu seberapa halus scale dibagi (satu untuk seluruh tensor, satu per baris, atau satu per token), sama dengan simulasi kita, walaupun detailnya tidak identik: torchao memakai 8 bit simetris, sedangkan simulasi kita 7 bit dengan zero point. Di baris pertama, resep ini hanya diterapkan ke 48 proyeksi blok.

Sel ini bukan listing buku, jadi error di sini adalah masalah sungguhan. Penyebab yang paling mungkin adalah versi torchao: untuk torch 2.11 sampai 2.14, baris versi di sel persiapan seharusnya menyebut 0.18.0. Jangan lewati sel ini, karena `ao_int8` dipakai lagi sampai bagian distilasi.

Kembali ke hasilnya. Di run referensi, varian pertama, yang hanya menyasar 48 blok, memberi loss 3.767, hanya +0.005 dari model asli, dengan checkpoint 243.5 MB. Bandingkan dengan `torch.ao` per-channel di ukuran yang praktis sama: +0.324. Menurut Under the hood 1, perbedaan utama keduanya adalah cara menghitung scale aktivasi: satu untuk seluruh tensor, atau satu untuk setiap token.

Baris kedua menambahkan embedding. `wte` di-quantize dengan `Int8WeightOnlyConfig`: bobotnya disimpan dalam int8, tapi aktivasi yang lewat tidak disentuh, sehingga outlier di input `lm_head` aman. Karena `quantize_` memutus ikatan *tied*, `lm_head` lalu diarahkan kembali ke `wte` yang sudah ter-quantisasi. Hasilnya 127.9 MB: **74% lebih kecil dari model asli, dengan loss hanya +0.018 nats** pada penggaris kita, dan ikatan *tied* tetap utuh. Dari 536 MB dengan loss yang rusak ke 128 MB dengan loss yang hampir sama. Yang membuatnya mungkin adalah dua hal yang tidak ada di listing buku: tahu di mana outlier berada, dan tahu lapisan mana yang aktivasinya tidak boleh di-quantize.

Sekarang lirik kolom milidetik. Sel ini tidak mengukur float32, jadi kedua angkanya hanya bisa dibandingkan satu sama lain. Di run referensi, varian dengan embedding int8 lebih lambat (354 ms) daripada varian blok saja (236 ms); di CPU lain selisihnya bisa lain, bahkan hilang. Dugaan kita: matriks embedding 50,257 x 768 harus dikembalikan ke float (*dequantize*) setiap kali dipakai, dan di CPU itu pekerjaan tambahan tersebut memakan keuntungan dari tempat lain; penyebabnya tidak diukur langsung di sini. Jadi byte dan milidetik adalah dua tujuan yang berbeda, dan Under the hood 4 akan menimbang keduanya berdampingan, bersama float32 di sel yang sama.

Static quantization berbeda di satu titik. Scale aktivasi tidak dihitung saat inferensi, tapi sekali saja, sebelum model dipakai, dari sejumlah data contoh, lalu dibekukan. Proses ini disebut *kalibrasi*. Yang mencatat rentang nilai selama kalibrasi adalah *observer*, modul kecil yang ditempelkan pada keluaran lapisan dan merekam nilai-nilai yang lewat, misalnya nilai terkecil dan terbesarnya. Rentang keluaran satu lapisan sekaligus menjadi rentang input lapisan sesudahnya. Karena scale-nya sudah tetap, inferensi tidak perlu lagi menghitung rentang, dan karena itu buku menjanjikan cara ini sebagai yang paling cepat.

Observer mana yang dipakai ditentukan oleh *qconfig*, resep kecil berisi dua pilihan: observer untuk aktivasi, dan observer untuk bobot. Resep itu ditempelkan ke model, atau ke lapisan tertentu, lewat atribut `.qconfig`, lalu diwariskan ke semua submodulnya. Detail pewarisan ini nanti menjadi sumber masalah di bagian PTQ.

Buku mendemonstrasikannya bukan pada LLM, tapi pada model kecil `SimpleModel`: dua lapisan linear (784 ke 256 ke 10) dengan ReLU di antaranya. Sesudah qconfig dipasang, alurnya tiga langkah. `prepare` membaca qconfig dan menyisipkan observer, 100 input acak `torch.randn` dilewatkan sebagai data kalibrasi, lalu `convert` mengganti lapisan-lapisannya dengan versi int8. `get_default_qconfig('fbgemm')` di listing adalah resep bawaan untuk CPU x86. Perhatikan baris terakhir listing-nya.

In [9]:
%%boleh_gagal
import torch
import torch.nn as nn
import torch.quantization
# Define a simple model
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(784, 256)
        self.relu = nn.ReLU()
        self.out = nn.Linear(256, 10)
    def forward(self, x):
        x = self.relu(self.fc(x))
        return self.out(x)
# Create and prepare model for static quantization
model_fp32 = SimpleModel()
model_fp32.eval()
model_fp32.qconfig = torch.quantization.get_default_qconfig(
    'fbgemm')
prepared_model = torch.quantization.prepare(model_fp32)
# Calibration step: run representative data through the model
# (This example uses random data; replace with real samples)
for _ in range(100):
    sample_input = torch.randn(1, 784)
    prepared_model(sample_input)
# Convert to quantized version
quantized_model = torch.quantization.convert(prepared_model)
# Model is now statically quantized and ready for inference
print(quantized_model


  File "<listing-2>", line 28
    print(quantized_model
         ^
SyntaxError: '(' was never closed
[sel ini berhenti di SyntaxError; notebook tetap lanjut ke sel berikutnya]


Sel ini bahkan tidak sempat berjalan. Python sudah menolaknya ketika *membaca* kode, sebelum satu baris pun dieksekusi: `SyntaxError: '(' was never closed`, menunjuk baris terakhir, `print(quantized_model`, yang kehilangan kurung tutup. Akibatnya `SimpleModel` pun belum terdefinisi. Satu karakter memperbaikinya, dan sel berikut menjalankan listing yang sama dengan `)` di tempatnya.

In [10]:
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(784, 256)
        self.relu = nn.ReLU()
        self.out = nn.Linear(256, 10)
    def forward(self, x):
        x = self.relu(self.fc(x))
        return self.out(x)
model_fp32 = SimpleModel()
model_fp32.eval()
model_fp32.qconfig = torch.quantization.get_default_qconfig(
    'fbgemm')
prepared_model = torch.quantization.prepare(model_fp32)
for _ in range(100):
    sample_input = torch.randn(1, 784)
    prepared_model(sample_input)
quantized_model = torch.quantization.convert(prepared_model)
print(quantized_model)                                    # perbaikan #4: kurung ditutup


SimpleModel(
  (fc): QuantizedLinear(in_features=784, out_features=256, scale=0.03519655764102936, zero_point=65, qscheme=torch.per_channel_affine)
  (relu): ReLU()
  (out): QuantizedLinear(in_features=256, out_features=10, scale=0.011033362708985806, zero_point=63, qscheme=torch.per_channel_affine)
)


Kedua `nn.Linear` kini menjadi `QuantizedLinear`. Bobotnya memakai `per_channel_affine`, satu scale per baris, karena qconfig bawaan `'fbgemm'` memakai observer per-channel untuk bobot. Untuk aktivasi ia memakai observer jenis Histogram, yang akan kita bandingkan dengan jenis lain di Under the hood 2. Pasangan `scale` dan `zero_point` yang tercetak di setiap lapisan adalah parameter untuk *output* lapisan itu, dibekukan dari 100 sampel kalibrasi. Listing ini tidak memasang seed sendiri, jadi angka-angka itu bergantung pada keadaan generator acak saat sel dijalankan; kalau sel ini dijalankan ulang, angkanya ikut berubah.

Komentar terakhir listing menyebut model ini *"ready for inference"*. Tapi listing berhenti di `print`, dan model-nya tidak pernah dipanggil. Mari kita panggil, dengan input yang bentuknya sama dengan data kalibrasi.

In [11]:
%%boleh_gagal
quantized_model(torch.randn(1, 784))


Traceback (most recent call last):
  File "<listing-3>", line 1, in <module>
    quantized_model(torch.randn(1, 784))
    ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py", line 1779, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py", line 1790, in _call_impl
    return forward_call(*args, **kwargs)
  File "/tmp/ipykernel_584/3410496562.py", line 8, in forward
    x = self.relu(self.fc(x))
                  ~~~~~~~^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py", line 1779, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py", line 1790, in _call_impl
    return forward_call(*args, **kwargs)
  File "/usr/local/lib/python3.13/dist

Begitu model itu dipanggil, muncul `NotImplementedError: Could not run 'quantized::linear' with arguments from the 'CPU' backend`, diikuti daftar nama backend. Isi daftar itu bisa berbeda antarversi torch, tapi petunjuknya ada di sana: `quantized::linear` hanya terdaftar untuk backend *Quantized*, yaitu untuk tensor yang *sudah* ter-quantisasi (`QuantizedCPU`, dan di build torch dengan CUDA juga `QuantizedCUDA`). Yang kita berikan adalah tensor float biasa di CPU, dan untuk itu tidak ada kernelnya.

Dalam static quantization gaya *eager* yang dipakai buku, model tidak tahu di mana float berhenti dan int8 dimulai. Batas itu harus ditandai sendiri. `QuantStub` di pintu masuk mengubah input float menjadi `quint8` (bilangan bulat 8 bit tanpa tanda, 0 sampai 255) dengan scale hasil kalibrasi, dan `DeQuantStub` di pintu keluar mengembalikannya ke float. `SimpleModel` buku tidak punya keduanya, sehingga `convert` menghasilkan model yang lapisan pertamanya menolak input float apa pun. Selama listing hanya mencetak model, kesalahan ini tidak kelihatan.

Perbaikannya empat baris: dua di `__init__` dan dua di `forward`. Kalibrasinya tetap seperti buku, 100 sampel `torch.randn`, kali ini dengan seed tetap supaya angkanya bisa diulang. Sesudah konversi, kita bandingkan output model int8 dengan model float32 pada 256 input uji, beserta ukuran `state_dict` keduanya.

In [12]:
class SimpleModelQ(nn.Module):                             # perbaikan #5: pintu masuk/keluar float <-> int8
    def __init__(self):
        super().__init__()
        self.quant = torch.quantization.QuantStub()
        self.fc = nn.Linear(784, 256)
        self.relu = nn.ReLU()
        self.out = nn.Linear(256, 10)
        self.dequant = torch.quantization.DeQuantStub()
    def forward(self, x):
        x = self.quant(x)
        x = self.relu(self.fc(x))
        return self.dequant(self.out(x))

torch.manual_seed(0)
m_fp32 = SimpleModelQ().eval()
m_fp32.qconfig = torch.quantization.get_default_qconfig('fbgemm')
m_prep = torch.quantization.prepare(copy.deepcopy(m_fp32))
for _ in range(100):                                       # kalibrasi persis buku: torch.randn
    m_prep(torch.randn(1, 784))
m_int8 = torch.quantization.convert(m_prep)
print(m_int8)
x_test = torch.randn(256, 784)
with torch.no_grad():
    y32, y8 = m_fp32(x_test), m_int8(x_test)
print(f"\n256 input uji: max |y_fp32 - y_int8| = {(y32 - y8).abs().max():.4f} (rata-rata |y_fp32| = {y32.abs().mean():.4f}) "
      f"| argmax sama: {(y32.argmax(1) == y8.argmax(1)).float().mean():.1%}")
print(f"state_dict: {nbytes(m_fp32)/1e3:.0f} KB -> {nbytes(m_int8)/1e3:.0f} KB")


SimpleModelQ(
  (quant): Quantize(scale=tensor([0.0634]), zero_point=tensor([64]), dtype=torch.quint8)
  (fc): QuantizedLinear(in_features=784, out_features=256, scale=0.03787488490343094, zero_point=59, qscheme=torch.per_channel_affine)
  (relu): ReLU()
  (out): QuantizedLinear(in_features=256, out_features=10, scale=0.011293431743979454, zero_point=66, qscheme=torch.per_channel_affine)
  (dequant): DeQuantize()
)

256 input uji: max |y_fp32 - y_int8| = 0.0530 (rata-rata |y_fp32| = 0.1851) | argmax sama: 97.3%
state_dict: 817 KB -> 214 KB


Sekarang model int8-nya benar-benar bekerja. Di output run referensi, `Quantize` di pintu masuk membawa scale 0.0634 dengan zero point 64. Kalau dihitung balik, rentang yang bisa diwakilinya kira-kira -4.06 sampai 3.99 dengan 128 nilai, pas untuk input `torch.randn` yang hampir semua nilainya berada di antara -4 dan 4. Zero point 64 jatuh di tengah rentang 0 sampai 127, karena input itu tersebar di sekitar nol. Hanya 128 nilai, bukan 256, karena observer di qconfig ini juga memakai `reduce_range`.

Pada 256 input uji, selisih terbesar antara output int8 dan output float32 adalah 0.053, sementara rata-rata besarnya output 0.185. Jadi di titik terburuknya, di antara 2,560 angka output (256 input x 10 kelas), pembulatan menggeser hasil hampir 30% dari besar output rata-rata. Untuk model klasifikasi, angka berikutnya lebih penting: 97.3% prediksi (kelas dengan skor tertinggi) sama dengan model float32, artinya hanya sekitar 7 dari 256 input yang kelas tebakannya berubah. `state_dict` menyusut dari 817 ke 214 KB, turun 74%, mendekati rasio 4 byte ke 1 byte; sebagian sisanya adalah bias dan scale yang tetap disimpan dalam float.

Di model sekecil ini, klaim penghematan sekitar 75% dari buku terpenuhi dengan mudah: tidak ada embedding, dan data uji berasal dari distribusi yang sama dengan data kalibrasi. Di GPT-2 kedua kemudahan itu tidak ada, dan bagian PTQ berikut memperlihatkan akibatnya.

### PTQ

*Post-training quantization* (PTQ) berarti meng-quantize model yang sudah selesai dilatih, tanpa melatihnya lagi. Dynamic quantization yang kita pakai tadi termasuk PTQ, begitu juga static quantization. Untuk yang statis, prosa buku sendiri menegaskan bahwa PTQ *"often require a calibration step on a representative dataset to determine optimal quantization parameters"*, yaitu kalibrasi dengan data yang mewakili data sesungguhnya.

Listing PTQ buku memakai urutan yang sama dengan `SimpleModel`: pasang qconfig, `prepare`, lalu `convert`. Bedanya, model-nya kali ini dimaksudkan sebagai LLM pretrained, dan placeholder-nya kembali `model = ...`.

In [13]:
%%boleh_gagal
import torch
import torch.quantization as quant
# Load pre-trained model
model = ...
# Convert model to quantization-ready state
model.eval()
model.qconfig = torch.quantization.default_qconfig
# Prepare for static quantization
model_prepared = quant.prepare(model)
# Apply quantization
model_quantized = quant.convert(model_prepared)


Traceback (most recent call last):
  File "<listing-4>", line 6, in <module>
    model.eval()
    ^^^^^^^^^^
AttributeError: 'ellipsis' object has no attribute 'eval'
[sel ini berhenti di AttributeError; notebook tetap lanjut ke sel berikutnya]


Listing PTQ jatuh di lubang yang sama dengan listing pertama, malah lebih awal: kali ini listing sendiri yang memanggil `model.eval()`, jadi `AttributeError` yang sama muncul di baris itu. Komentar listing menyebut *"Load pre-trained model"*, jadi kita isi dengan model pretrained yang paling relevan, salinan GPT-2 kita, lalu menjalankan sisa listing apa adanya.

In [14]:
%%boleh_gagal
model = copy.deepcopy(gpt2)                                # perbaikan #6: placeholder diisi GPT-2 pretrained
# Convert model to quantization-ready state
model.eval()
model.qconfig = torch.quantization.default_qconfig
# Prepare for static quantization
model_prepared = quant.prepare(model)
# Apply quantization
model_quantized = quant.convert(model_prepared)


Traceback (most recent call last):
  File "<listing-5>", line 8, in <module>
    model_quantized = quant.convert(model_prepared)
  File "/usr/lib/python3.13/warnings.py", line 637, in wrapper
    return arg(*args, **kwargs)
  File "/usr/local/lib/python3.13/dist-packages/torch/ao/quantization/quantize.py", line 669, in convert
    _convert(
    ~~~~~~~~^
        module,
        ^^^^^^^
    ...<4 lines>...
        use_precomputed_fake_quant=use_precomputed_fake_quant,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/torch/ao/quantization/quantize.py", line 726, in _convert
    _convert(
    ~~~~~~~~^
        mod,
        ^^^^
    ...<4 lines>...
        use_precomputed_fake_quant=use_precomputed_fake_quant,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/torch/ao/quantization/quantize.py", line 734, in _convert
    reassign[name] = swap_mod

Placeholder sudah terisi, dan listing kini tersandung di langkah berikutnya: `convert` berhenti dengan `AssertionError: Embedding quantization is only supported with float_qparams_weight_only_qconfig.` Penyebabnya baris `model.qconfig = ...` yang dipasang di akar model. Seperti kita lihat di bagian static quantization, qconfig diwariskan ke *setiap* submodul, termasuk embedding `wte` dan `wpe`, dan `convert` menolak meng-quantize embedding dengan qconfig statis biasa. Embedding butuh skema khusus yang hanya meng-quantize bobotnya.

Seandainya langkah itu lolos pun, kita bisa menduga apa yang terjadi berikutnya. GPT-2 tidak punya satu `QuantStub` pun, jadi lapisan yang ter-quantisasi akan menerima input float dan gagal persis seperti `SimpleModel` tadi. Ke-48 `Conv1D` juga tetap tidak terlihat.

Perbaikan yang tetap setia pada maksud listing terdiri dari tiga langkah. Pakai `g`, GPT-2 versi `nn.Linear` yang kita buat tadi. Bungkus setiap proyeksi blok dengan `QWrap`, pembungkus kecil berisi `QuantStub`, lapisan aslinya, dan `DeQuantStub`, sehingga embedding, LayerNorm, dan GELU tetap float. Lalu pasang qconfig buku *hanya* di ke-48 pembungkus itu. Sesudahnya urutan buku dijalankan apa adanya: `prepare`, lalu langsung `convert`, tanpa kalibrasi, karena listing-nya memang tidak memuat langkah itu.

Baris pertama sel berikut menghapus `model` dan `model_prepared`. Keduanya salinan GPT-2 yang masih tersisa dari listing yang gagal tadi: nama yang terbentuk sebelum baris yang error tetap ada.

In [15]:
del model, model_prepared; gc.collect()                    # knob memori: dua salinan GPT-2 dari sel gagal dilepas

class QWrap(nn.Module):                                    # perbaikan #7: stub di sekeliling tiap Linear; embedding/LN/GELU tetap float
    def __init__(self, lin):
        super().__init__()
        self.quant = torch.quantization.QuantStub(); self.lin = lin; self.dequant = torch.quantization.DeQuantStub()
    def forward(self, x):
        return self.dequant(self.lin(self.quant(x)))

def wrap_blocks(base, qconfig):
    m = copy.deepcopy(base)
    for blk in m.transformer.h:
        for parent, names in ((blk.attn, ("c_attn", "c_proj")), (blk.mlp, ("c_fc", "c_proj"))):
            for nm in names:
                w = QWrap(getattr(parent, nm)); w.qconfig = qconfig; setattr(parent, nm, w)
    return m.eval()

model = wrap_blocks(g, torch.quantization.default_qconfig)   # qconfig persis buku, hanya di 48 pembungkus
model_prepared = quant.prepare(model)
model_quantized = quant.convert(model_prepared)              # urutan persis buku: prepare -> convert, tanpa kalibrasi
st = model_quantized.transformer.h[0].attn.c_attn.quant
print(f"skala input h0.attn.c_attn: scale = {st.scale.item()}, zero_point = {st.zero_point.item()}")
print(f"val loss: {base_val:.3f} -> {eval_gpt2(model_quantized):.3f}")


/usr/local/lib/python3.13/dist-packages/torch/ao/quantization/observer.py:368: UserWarning: must run observer before calling calculate_qparams. Returning default values.
  if not check_min_max_valid(min_val, max_val):


skala input h0.attn.c_attn: scale = 1.0, zero_point = 0
val loss: 3.762 -> 8.079


Model hasil konversi kini bisa dipanggil, tapi hampir tidak berguna: di run referensi loss-nya naik dari 3.762 ke 8.079. Penyebabnya terlihat di dua tempat. Scale input `h0.attn.c_attn` bernilai 1.0 dengan zero point 0, dan stderr membawa peringatan dari torch sendiri: *"must run observer before calling calculate_qparams. Returning default values."*

Listing buku melompat dari `prepare` langsung ke `convert`. Observer yang disisipkan `prepare` tidak pernah melihat satu aktivasi pun, sehingga setiap scale jatuh ke nilai bawaan, 1.0. Bayangkan akibatnya pada input `c_attn` di layer 0, yaitu keluaran LayerNorm `ln_1`. Menurut anatomi di Under the hood 1, median nilai absolutnya hanya sekitar 0.073. Dengan jarak antarnilai 1.0, hampir semua nilai itu dibulatkan ke nol. Lebih buruk lagi, `quint8` hanya punya bilangan 0 sampai 255, dan dengan zero point 0 tidak ada tempat untuk angka negatif: semua nilai negatif ikut terpotong menjadi nol. Prosa buku menyebut PTQ statis sering membutuhkan kalibrasi; listing-nya justru tidak memuat langkah itu sama sekali.

### Under the hood 2: kalibrasi

Seberapa penting kalibrasi, dan apakah datanya harus teks sungguhan? Sel berikut (UTH-2) membandingkan tiga pilihan data kalibrasi. Yang pertama tanpa kalibrasi, mengikuti urutan listing buku. Yang kedua token acak, padanan `torch.randn` buku untuk model bahasa: input GPT-2 bukan vektor float melainkan nomor token, jadi padanannya adalah 8 x 128 nomor token yang diambil acak dari 50,257 kosakata. Yang ketiga teks asli, 8 potongan 128 token dari bagian *train* wikitext-2, bukan dari teks validasi yang menjadi penggaris kita.

Setiap pilihan dicoba dengan dua jenis observer aktivasi. *MinMax* mencatat nilai terkecil dan terbesar yang pernah lewat, lalu memakai rentang itu apa adanya, sehingga ia wajib menjangkau outlier terjauh. *Histogram* merekam sebaran nilai, lalu memilih rentang yang meminimalkan galat kuadrat akibat pembulatan, walaupun itu berarti memotong segelintir nilai ekstrem; observer inilah bawaan qconfig `'fbgemm'` dan `'x86'`. Bobot di semua varian memakai per-channel. Di bagian akhir, sel ini juga mencetak dua angka pembanding dari bagian dynamic quantization.

In [16]:
# === UNDER THE HOOD (UTH-2): kalibrasi PTQ statis -- tanpa, token acak, teks riil; MinMax vs Histogram ===
from torch.ao.quantization import QConfig, MinMaxObserver, HistogramObserver, PerChannelMinMaxObserver
del model, model_prepared, model_quantized; gc.collect()   # knob memori: tiga salinan GPT-2 dilepas (nama `model` ikut hilang)
calib_real = ft_train[:8]                                  # 8 x 128 token teks wikitext-train (bukan penggaris val)
calib_rand = torch.randint(0, 50257, (8, 128), generator=torch.Generator().manual_seed(0))   # padanan torch.randn buku untuk LM
w_obs = PerChannelMinMaxObserver.with_args(dtype=torch.qint8, qscheme=torch.per_channel_symmetric)
observers = {"MinMax": QConfig(activation=MinMaxObserver.with_args(reduce_range=True), weight=w_obs),
             "Histogram": QConfig(activation=HistogramObserver.with_args(reduce_range=True), weight=w_obs)}  # Histogram = default x86

def static_ptq(qconfig, calib):
    p = quant.prepare(wrap_blocks(g, qconfig))
    if calib is not None:
        with torch.no_grad():
            for i in range(0, len(calib), 2): p(input_ids=calib[i:i + 2])
    return quant.convert(p)

print(f"{'kalibrasi':<22} {'observer':<10} {'skala h0.c_attn':>15} {'val loss':>9} {'delta':>7}")
uth2 = {}
for cname, calib in (("tanpa (urutan buku)", None), ("token acak", calib_rand), ("teks riil", calib_real)):
    for oname, qc in observers.items():
        q = static_ptq(qc, calib)
        v = eval_gpt2(q); sc = q.transformer.h[0].attn.c_attn.quant.scale.item()
        uth2[(cname, oname)] = v
        print(f"{cname:<22} {oname:<10} {sc:>15.5f} {v:>9.3f} {v - base_val:>+7.3f}")
        del q; gc.collect()
print(f"\nreferensi -- dynamic per-channel torch.ao (48 blok): {dyn_rows['per-channel, 48 blok (lm_head fp32)'][0]:.3f} "
      f"| torchao W8A8 per-token (48 blok): {ao_rows['torchao W8A8 per-token (48 blok)'][0]:.3f}")


kalibrasi              observer   skala h0.c_attn  val loss   delta
tanpa (urutan buku)    MinMax             1.00000     8.080  +4.318


/usr/local/lib/python3.13/dist-packages/torch/ao/nn/quantized/modules/__init__.py:133: UserWarning: must run observer before calling calculate_qparams.                                    Returning default scale and zero point 
  scale, zero_point = mod.activation_post_process.calculate_qparams()
/usr/local/lib/python3.13/dist-packages/torch/ao/nn/quantized/modules/linear.py:334: UserWarning: must run observer before calling calculate_qparams.                                    Returning default scale and zero point 
  act_scale, act_zp = activation_post_process.calculate_qparams()


tanpa (urutan buku)    Histogram          1.00000     8.080  +4.318
token acak             MinMax             0.01247     5.336  +1.574
token acak             Histogram          0.00880     4.853  +1.091
teks riil              MinMax             0.01245     5.340  +1.578
teks riil              Histogram          0.01065     4.760  +0.998

referensi -- dynamic per-channel torch.ao (48 blok): 4.071 | torchao W8A8 per-token (48 blok): 3.767


Mulai dari dua baris teratas, yang mengikuti urutan listing buku. Tanpa satu pun forward kalibrasi, kedua observer menyerah ke scale 1.0, dan di run referensi loss tertahan di 8.080 untuk keduanya: inilah model yang dihasilkan listing PTQ buku bila dijalankan pada model sungguhan. Selisih 0.001 dengan 8.079 di sel listing tadi datang dari bobot, satu-satunya pengaturan yang berbeda: di sini bobot memakai scale per baris, sedangkan `default_qconfig` buku memakai satu scale per tensor. Peringatan *must run observer* di stderr sel ini juga berasal dari dua baris tanpa kalibrasi itu. Kalibrasi jelas bukan langkah yang boleh dilewati.

Begitu kalibrasi dijalankan, yang paling menentukan ternyata jenis observer, bukan jenis data. Histogram unggul sekitar 0.55 sampai 0.6 nats atas MinMax: 4.827 lawan 5.375 dengan token acak, dan 4.762 lawan 5.378 dengan teks asli. Kolom scale memperlihatkan alasannya. MinMax memberi `h0.c_attn` scale sekitar 0.0125, sedangkan Histogram 0.0088 sampai 0.0107. Karena Histogram rela memotong segelintir nilai ekstrem, jarak antarnilainya lebih rapat untuk semua nilai lain; ini prinsip yang sama dengan contoh bobot 5.0 di awal bab.

Jenis data, sebaliknya, hampir tidak berpengaruh: **token acak hampir sama baiknya dengan teks asli** untuk kalibrasi. Dengan Histogram selisihnya 0.065 nats (4.827 lawan 4.762), dan dengan MinMax keduanya praktis sama (5.375 lawan 5.378). Angka ini paling mudah dijelaskan kalau outlier GPT-2 bersifat struktural: muncul di dimensi yang tetap apa pun teks yang masuk, sehingga rentang yang dikalibrasi dari deretan token acak hampir sama dengan rentang dari Wikipedia. Scale MinMax di `h0.c_attn` cocok dengan dugaan itu (0.01247 dari token acak, 0.01245 dari teks asli), walaupun itu baru satu lapisan.

Meski begitu, PTQ statis yang paling baik (+1.000) masih jauh di belakang PTQ dinamis: +0.324 untuk `torch.ao` per-channel, dan +0.005 untuk torchao per-token. Ada dua perbedaan yang kemungkinan besar berperan, walaupun eksperimen ini tidak memisahkannya. Scale aktivasinya beku, satu angka per lapisan untuk semua token, padahal besar outlier bisa berbeda dari token ke token; torchao, yang menghitung scale untuk setiap token, hampir tidak merusak sama sekali. Selain itu, output setiap lapisan ikut dibulatkan ke int8 sebelum diteruskan, jadi ada satu pembulatan tambahan di setiap lapisan. Prosa buku menyebut PTQ *"generally maintains accuracy within acceptable limits"*. Untuk PTQ statis dengan API buku di GPT-2, varian terbaik dari enam yang kita coba masih kehilangan sekitar 1 nats, artinya peluang untuk token yang benar tinggal sekitar sepertiganya.

### QAT

PTQ meng-quantize model yang sudah jadi; model itu tidak pernah tahu bahwa angkanya akan dibulatkan. *Quantization-aware training* (QAT) mengambil jalan lain: pembulatan sudah disimulasikan selama training, sehingga model bisa belajar menyesuaikan diri dengannya. Alatnya fake quantization yang kita pakai di Under the hood 1. Di setiap forward, bobot dan aktivasi di-quantize lalu dikembalikan ke float, dan loss dihitung dari model yang sudah "merasakan" pembulatan itu.

Ada satu kesulitan. Pembulatan adalah fungsi bertangga: turunannya nol hampir di mana-mana, sehingga gradien tidak bisa mengalir melewatinya. QAT mengatasinya dengan *straight-through estimator* (STE). Saat backward, langkah pembulatan diperlakukan seolah tidak ada, dan gradien diteruskan apa adanya ke bobot float di belakangnya, selama nilainya masih di dalam rentang yang bisa diwakili. Bobot float itulah yang terus diperbarui, dan baru di-quantize sungguhan setelah training selesai.

Buku menjanjikan akurasi yang lebih baik daripada PTQ, *"particularly for more complex tasks"*. Listing-nya hanya kerangka. Kita jalankan dengan `%%boleh_gagal --fresh`, yaitu di namespace kosong seperti kernel baru, supaya hasilnya tidak bergantung pada sel mana yang kebetulan pernah dijalankan sebelumnya.

In [17]:
%%boleh_gagal --fresh
import torch.quantization as quant
# Set up QAT
model.train()
model.qconfig = torch.quantization.get_default_qat_qconfig('fbgemm')
# Prepare for QAT
model_prepared = quant.prepare_qat(model)
# Training loop (for simplicity, only showing initialization)
for epoch in range(num_epochs):
    train_one_epoch(model_prepared, train_loader, optimizer)
    validate(model_prepared, val_loader)
# Convert to quantized version
model_quantized = quant.convert(model_prepared.eval())


Traceback (most recent call last):
  File "<listing-6>", line 3, in <module>
    model.train()
    ^^^^^
NameError: name 'model' is not defined
[sel ini berhenti di NameError; notebook tetap lanjut ke sel berikutnya]


Listing ini berhenti di baris ketiganya dengan `NameError: name 'model' is not defined`. Kerangka QAT buku memang tidak pernah mendefinisikan model-nya. Di belakang `model` masih antre enam nama lain yang juga tidak pernah ada: `num_epochs`, `train_one_epoch`, `train_loader`, `optimizer`, `validate`, dan `val_loader`. Bahkan `torch` tidak pernah di-import. Baris pertama hanya membuat alias `quant`, lalu baris keempat memakai `torch.quantization.get_default_qat_qconfig`; di namespace kosong, baris itu pun akan gagal seandainya `model` ada. Di notebook biasa kesalahan ini tidak terlihat, karena sel-sel sebelumnya sudah meng-import `torch`.

Untuk mengisi kerangka itu kita butuh model yang bisa dilatih dalam hitungan menit di CPU, dan GPT-2 124M terlalu besar untuk itu. Jadi kita pakai GPT-2 mini: arsitekturnya sama, tapi hanya 2 layer dengan vektor 64 dimensi, dan dilatih dari nol. Korpusnya dibuat dari 20,000 baris pertama data training wikitext-2, ditokenisasi dengan tokenizer GPT-2, lalu kosakatanya dipangkas ke 4,095 token yang paling sering muncul, ditambah satu token `<unk>` untuk semua token lain, sehingga totalnya 4,096. Output dan komentar kode menyebut teks ini *korpus proxy*, yaitu teks pengganti yang cukup kecil untuk melatih model mini di CPU. Komentar kodenya juga merujuk resep yang sama di bagian lain seri ini, tapi semua yang dibutuhkan sudah ada di sel ini. Model mini dilatih 2,000 langkah dalam float32, dan checkpoint itulah titik awal semua percobaan QAT dan PTQ berikutnya.

Sebelum memperbaiki apa pun, satu pertanyaan sensus: kalau resep buku, yaitu qconfig QAT di akar model lalu `prepare_qat`, diterapkan apa adanya pada GPT-2 mini, lapisan mana yang benar-benar diganti dengan versi QAT, yang bobotnya ikut "merasakan" pembulatan selama training? Sel berikut menjawabnya, lalu menjalankan versi perbaikannya: `conv1d_to_linear`, pembungkus `QWrap` di setiap proyeksi, qconfig QAT hanya di pembungkus, kemudian 300 langkah QAT dari checkpoint float32. Qconfig-nya memakai `'x86'`, sesuai engine yang aktif, menggantikan `'fbgemm'` di buku.

Hasil QAT itu dibandingkan dengan dua pembanding: PTQ int8 tanpa training tambahan, dan model float32 yang dilatih 300 langkah yang sama tanpa quantization. Pembanding kedua disebut *kelompok kontrol*: ia mendapat semua yang didapat model QAT, kecuali satu hal yang ingin kita uji, yaitu pembulatan yang ikut disimulasikan selama training.

In [18]:
# Korpus proxy standar seri + model mini (resep Lampiran B)
SEQ_LEN, VOCAB = 64, 4096
tok_mini = AutoTokenizer.from_pretrained("gpt2")
texts = [t for t in wiki_train["text"][:20000] if t.strip()]
raw_ids = []
for out in tok_mini(texts)["input_ids"]:
    raw_ids.extend(out + [tok_mini.eos_token_id])
freq = collections.Counter(raw_ids)
remap = {t: i + 1 for i, (t, _) in enumerate(freq.most_common(VOCAB - 1))}   # id 0 = <unk>
ids = torch.tensor([remap.get(t, 0) for t in raw_ids])
blocks = ids[:len(ids) // SEQ_LEN * SEQ_LEN].view(-1, SEQ_LEN)
perm = torch.randperm(len(blocks), generator=torch.Generator().manual_seed(0))
val_blocks, tr_blocks = blocks[perm[:256]], blocks[perm[256:]]
print(f"korpus proxy: {len(raw_ids):,} token | cakupan vocab {VOCAB}: {(ids > 0).float().mean():.1%} "
      f"| blok train {len(tr_blocks)} / val {len(val_blocks)}")

def make_mini(dropout=0.0):
    return GPT2LMHeadModel(GPT2Config(vocab_size=VOCAB, n_positions=SEQ_LEN, n_embd=64, n_layer=2, n_head=2,
        resid_pdrop=dropout, embd_pdrop=dropout, attn_pdrop=dropout, bos_token_id=0, eos_token_id=0))

def eval_mini(m):
    m.eval()
    with torch.no_grad():
        return float(np.mean([m(input_ids=val_blocks[i:i + 32], labels=val_blocks[i:i + 32]).loss.item()
                              for i in range(0, len(val_blocks), 32)]))

def train_mini(m, steps, start=0, lr=1e-3, bs=8):
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=0.0)
    m.train(); N = len(tr_blocks)
    for s in range(start, start + steps):
        i = (s * bs) % (N - bs); xb = tr_blocks[i:i + bs]          # batch diindeks dari step absolut
        loss = m(input_ids=xb, labels=xb).loss
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    opt.zero_grad(set_to_none=True); del opt

torch.manual_seed(0)
mini = make_mini(); conv1d_to_linear(mini)
t0 = time.time(); train_mini(mini, 2000, lr=3e-3)              # knob: 2000 step lr 3e-3 -- mini yang cukup terlatih untuk diuji kuantisasinya
SD_MINI = copy.deepcopy(mini.state_dict()); mini_fp32 = eval_mini(mini)
print(f"mini fp32 (2 layer, 2000 step): val {mini_fp32:.4f} | {time.time() - t0:.0f}s")

# (a) resep buku apa adanya: qconfig QAT di root, prepare_qat pada GPT-2 mini asli (Conv1D)
book_qat = make_mini(); book_qat.train()
book_qat.qconfig = torch.quantization.get_default_qat_qconfig('fbgemm')
book_qat = quant.prepare_qat(book_qat)
qat_mods = [n for n, m in book_qat.named_modules() if type(m).__module__.startswith("torch.ao.nn.qat")]
print(f"(a) prepare_qat resep buku: modul QAT = {qat_mods} | Conv1D tak tersentuh = "
      f"{sum(isinstance(m, Conv1D) for m in book_qat.modules())}")
try:
    quant.convert(book_qat.eval())(input_ids=val_blocks[:2])
except Exception as e:
    print(f"    convert + forward model hasil resep buku -> {type(e).__name__}: {str(e)[:90]}")
del book_qat

# (b) diperbaiki: Conv1D -> Linear, stub per Linear, qconfig QAT hanya di pembungkus; 300 step QAT dari checkpoint fp32
def wrap_mini(sd, qconfig):
    m = make_mini(); conv1d_to_linear(m); m.load_state_dict(sd)
    for blk in m.transformer.h:
        for parent, names in ((blk.attn, ("c_attn", "c_proj")), (blk.mlp, ("c_fc", "c_proj"))):
            for nm in names:
                w = QWrap(getattr(parent, nm)); w.qconfig = qconfig; setattr(parent, nm, w)
    return m
qat = quant.prepare_qat(wrap_mini(SD_MINI, torch.quantization.get_default_qat_qconfig('x86')).train())
t0 = time.time(); train_mini(qat, 300, start=2000, lr=1e-3)
qat_int8 = quant.convert(qat.eval()); v_qat = eval_mini(qat_int8); t_qat = time.time() - t0
ptq = quant.prepare(wrap_mini(SD_MINI, torch.quantization.get_default_qconfig('x86')).eval())
with torch.no_grad():
    for i in range(0, 64, 8): ptq(input_ids=tr_blocks[i:i + 8])
v_ptq = eval_mini(quant.convert(ptq))
ctrl = make_mini(); conv1d_to_linear(ctrl); ctrl.load_state_dict(SD_MINI); train_mini(ctrl, 300, start=2000, lr=1e-3)
v_ctrl = eval_mini(ctrl)
print(f"(b) QAT int8 (+300 step, {t_qat:.0f}s): val {v_qat:.4f} | PTQ int8 tanpa latihan: {v_ptq:.4f} "
      f"| kontrol fp32 +300 step: {v_ctrl:.4f} | Linear hasil convert: {type(qat_int8.transformer.h[0].mlp.c_fc.lin).__module__}")


korpus proxy: 1,302,724 token | cakupan vocab 4096: 83.2% | blok train 20099 / val 256
mini fp32 (2 layer, 2000 step): val 4.3176 | 66s
(a) prepare_qat resep buku: modul QAT = ['lm_head'] | Conv1D tak tersentuh = 8
    convert + forward model hasil resep buku -> AssertionError: Embedding quantization is only supported with float_qparams_weight_only_qconfig.
(b) QAT int8 (+300 step, 15s): val 4.2563 | PTQ int8 tanpa latihan: 4.3196 | kontrol fp32 +300 step: 4.2547 | Linear hasil convert: torch.ao.nn.quantized.modules.linear


Korpus mini berisi sekitar 1.3 juta token, dan 4,095 token terpopuler mencakup 83.2% di antaranya; sisanya menjadi `<unk>`. Setelah 2,000 langkah, model mini float32 mencapai loss validasi 4.3097 di run referensi. Satuannya sama, nats, tapi penggarisnya berbeda dari GPT-2 (kosakatanya 4,096 token dan teks validasinya lain), jadi angka ini tidak bisa dibandingkan dengan 3.762. Sebagai pegangan, tebakan asal di antara 4,096 token memberi loss sekitar 8.3.

Sensusnya menjawab pertanyaan tadi dengan telak. `prepare_qat` mengganti tepat **satu** modul dengan versi QAT, yaitu `lm_head`, sementara 8 `Conv1D` tempat hampir seluruh perhitungan blok berlangsung tidak tersentuh. Jadi kalau resep "quantization-aware training" buku dipakai untuk training, hanya bobot satu lapisan yang ikut merasakan pembulatan, yaitu `lm_head`; bobot di blok-bloknya tidak. Begitu dicoba di-`convert`, resep yang sama menabrak tembok yang kita kenal dari bagian PTQ: `AssertionError` karena embedding.

Versi perbaikannya berjalan dari awal sampai akhir dan menghasilkan model int8 sungguhan yang bisa dipanggil; lapisannya kini modul dari `torch.ao.nn.quantized`. Loss-nya 4.2483, lebih baik daripada PTQ int8 tanpa training tambahan (4.3123). Sekilas ini tampak seperti kemenangan QAT. Tapi lihat angka ketiga: kelompok kontrol, model float32 yang dilatih 300 langkah yang sama tanpa quantization sama sekali, mencapai 4.2462, dan model QAT int8 hanya tertinggal 0.002 darinya. Jadi hampir seluruh keunggulan QAT atas PTQ di sini, sekitar 0.064 nats, tampaknya datang dari 300 langkah training tambahan, bukan dari pembulatan yang ikut disimulasikan selama training. Dengan satu seed, selisih 0.002 sendiri tidak berarti banyak; yang jelas, kelompok kontrol membaik hampir sebanyak model QAT.

Perhatikan juga baris PTQ: di model mini, PTQ statis int8 hampir tidak merusak (4.3123 lawan 4.3097), sangat berbeda dengan +1.0 di GPT-2 pretrained. Kemungkinan karena model mini yang baru dilatih 2,000 langkah belum punya outlier aktivasi sebesar GPT-2. (Waktu yang tercetak di sel ini milik mesin run referensi, dan angka waktu untuk QAT juga mencakup konversi dan evaluasi.)

Kalau berlatih sambil merasakan pembulatan punya nilai sendiri, nilainya harus terlihat ketika pembulatannya jauh lebih kasar daripada 8 bit. Itulah wilayah yang disebut buku: QAT unggul *"especially when targeting aggressive quantization (sub-8-bit)"*.

### Under the hood 3: QAT vs PTQ di bawah 8 bit

`torch.ao` tidak punya kernel CPU di bawah 8 bit untuk lapisan linear, jadi kita kembali ke fake quantization. Sel berikut (UTH-3) mengganti 8 proyeksi blok model mini dengan `FQLinear`: lapisan linear yang bobotnya di-fake-quantize per-channel secara simetris dengan jumlah bit tertentu, dengan gradien yang diteruskan lewat STE. Aktivasinya dibiarkan float32. Jadi berbeda dengan QAT int8 tadi, yang juga mensimulasikan pembulatan aktivasi, ini percobaan untuk bobot saja.

Dengan skema simetris ini, untuk b bit setiap bobot dibulatkan ke bilangan bulat dari -(2^(b-1) - 1) sampai 2^(b-1) - 1, dikali scale baris itu. Untuk 4 bit itu 15 nilai, untuk 3 bit 7 nilai, dan untuk 2 bit hanya 3 nilai per baris: -s, 0, dan +s, dengan s adalah scale baris tersebut. Kode `FQLinear` memang mengizinkan batas bawah -2^(b-1), tapi karena scale dihitung dari nilai absolut terbesar di baris itu, nilai tersebut tidak pernah tercapai.

Percobaannya dua tahap. Pertama, PTQ langsung dari checkpoint 2,000 langkah di 8, 4, 3, dan 2 bit. Kedua, untuk 4, 3, dan 2 bit, dua model dilatih 300 langkah lagi dengan seed dan batch yang sama. Kelompok kontrol dilatih dalam float32 lalu di-quantize sesudahnya, sedangkan model QAT dilatih dengan fake quantization yang sudah menyala. Kolom "untung QAT" adalah loss kontrol dikurangi loss QAT, sehingga angka positif berarti QAT lebih baik.

In [19]:
# === UNDER THE HOOD (UTH-3): QAT vs PTQ di bawah 8 bit -- dengan arm kontrol ===
class FQLinear(nn.Linear):                                 # fake-quant bobot per-channel simetris; gradien lewat STE torch.fake_quantize_*
    bits = None
    def forward(self, x):
        w = self.weight
        if self.bits:
            qmax = 2 ** (self.bits - 1) - 1
            scale = (w.detach().abs().amax(1) / qmax).clamp(min=1e-8)
            w = torch.fake_quantize_per_channel_affine(w, scale, torch.zeros_like(scale, dtype=torch.int32), 0, -qmax - 1, qmax)
        return F.linear(x, w, self.bias)

def fq_mini(sd):
    m = make_mini(); conv1d_to_linear(m)
    for blk in m.transformer.h:
        for parent, names in ((blk.attn, ("c_attn", "c_proj")), (blk.mlp, ("c_fc", "c_proj"))):
            for nm in names:
                lin = getattr(parent, nm); fq = FQLinear(lin.in_features, lin.out_features)
                fq.weight, fq.bias = lin.weight, lin.bias; setattr(parent, nm, fq)
    m.load_state_dict(sd); return m

def set_bits(m, b):
    for x in m.modules():
        if isinstance(x, FQLinear): x.bits = b

m = fq_mini(SD_MINI); ptq_row = {}
for b in (8, 4, 3, 2):
    set_bits(m, b); ptq_row[b] = eval_mini(m)
print(f"PTQ bobot 8 Linear blok (per-channel) dari checkpoint 2000 step -- fp32 {mini_fp32:.4f} | "
      + " | ".join(f"W{b} {ptq_row[b]:.4f}" for b in (8, 4, 3, 2)))
print(f"\n{'bit':>3} {'PTQ':>8} {'kontrol: fp32 +300 -> PTQ':>26} {'QAT +300':>9} {'untung QAT':>11}")
uth3 = {}
for b in (4, 3, 2):
    res = {}
    for arm in ("ctrl", "qat"):
        torch.manual_seed(0)
        mm = fq_mini(SD_MINI); set_bits(mm, b if arm == "qat" else None)
        train_mini(mm, 300, start=2000, lr=1e-3)
        set_bits(mm, b); res[arm] = eval_mini(mm); del mm
    uth3[b] = res
    print(f"{b:>3} {ptq_row[b]:>8.4f} {res['ctrl']:>26.4f} {res['qat']:>9.4f} {res['ctrl'] - res['qat']:>+11.4f}")
print(f"\nkontrol fp32 +300 step, tanpa kuantisasi: {v_ctrl:.4f}")


PTQ bobot 8 Linear blok (per-channel) dari checkpoint 2000 step -- fp32 4.3176 | W8 4.3176 | W4 4.3215 | W3 4.3416 | W2 4.8458

bit      PTQ  kontrol: fp32 +300 -> PTQ  QAT +300  untung QAT
  4   4.3215                     4.2575    4.2574     +0.0001
  3   4.3416                     4.2904    4.2663     +0.0242
  2   4.8458                     4.9431    4.3246     +0.6185

kontrol fp32 +300 step, tanpa kuantisasi: 4.2547


Tabel ini memisahkan dua hal yang tercampur dalam klaim buku: training tambahan, dan efek berlatih sambil merasakan pembulatan. Semua angkanya dari run referensi, dengan satu seed. Di runtime lain angka model mini bisa bergeser cukup jauh, terutama di baris 2 bit, jadi yang perlu dicocokkan adalah polanya: untung QAT yang besar di 2 bit, dan kecil di 4 dan 3 bit.

Baca baris PTQ dulu, yang diambil langsung dari checkpoint 2,000 langkah. Bobot 8 bit gratis (4.3097, sama dengan float32), 4 bit hampir gratis (+0.003), 3 bit +0.018, dan 2 bit runtuh (+0.572). Dengan hanya tiga nilai per baris, bobot tidak bisa lagi dibulatkan tanpa kehilangan banyak informasi.

Lalu kelompok kontrol, yang biasanya absen dari perbandingan QAT dan PTQ. Di 4 bit, kontrol (4.2526) dan QAT (4.2506) hanya berselisih 0.002 nats, dan di 3 bit 0.009. Selisih sekecil itu, dari satu seed, bisa saja berbalik arah kalau percobaannya diulang dengan seed lain. Hampir seluruh perbaikan dari PTQ awal di 4 bit, dari 4.313 ke sekitar 4.25, juga didapat oleh kontrol, jadi asalnya training tambahan.

Di 2 bit gambarnya berbalik total. Kontrol yang di-quantize sesudah training terpuruk di 5.007, sementara QAT bertahan di 4.322: **untung 0.685 nats**, dan hanya 0.076 nats lebih buruk daripada model float32 tanpa quantization sama sekali (4.2462). Ketika setiap baris hanya boleh memakai tiga nilai, model yang dilatih bersama pembulatan itu rupanya sempat menemukan bobot yang tetap bekerja; model yang baru dibulatkan sesudah training tidak punya kesempatan itu. Perhatikan juga bahwa kontrol 2 bit (5.007) malah lebih buruk daripada PTQ 2 bit langsung dari checkpoint 2,000 langkah (4.882). Training float tambahan memperbaiki model float32-nya, tetapi tidak membuat bobotnya lebih tahan dibulatkan ke tiga nilai. Dengan satu seed, kita hanya bisa mencatat hal itu, belum bisa menjelaskan sebabnya.

Jadi klaim buku benar, dengan satu ambang yang tidak disebut. Di model mini kita (dua layer, hanya bobot yang di-quantize, satu seed), QAT baru jelas menguntungkan di 2 bit. Di 4 dan 3 bit, keunggulannya tidak bisa dibedakan dari training tambahan biasa. Perbandingan QAT dengan PTQ tanpa kontrol yang dilatih sama lama sebenarnya mengukur dua hal sekaligus, lalu memberikan seluruh pujiannya kepada QAT.

## Mixed-precision quantization

Prosa buku mendefinisikan mixed-precision quantization dengan tepat: beberapa presisi dalam satu model. Lapisan yang kurang kritis disimpan di INT8, sedangkan lapisan yang sensitif tetap di FP16 atau FP32. Dari Under the hood 1 kita sudah punya calon kuat untuk lapisan yang sensitif, yaitu `lm_head` dengan outlier di input-nya.

Listing yang menyertainya justru memakai alat yang lain sama sekali, yaitu `autocast`. Kita jalankan apa adanya.

In [20]:
%%boleh_gagal
from torch.cuda.amp import autocast
# Mixed precision in LLM training or inference
model = ...
# Use FP16 where possible, fall back to FP32 for sensitive computations
with autocast():
    output = model(input_data)


<listing-7>:5: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Traceback (most recent call last):
  File "<listing-7>", line 6, in <module>
    output = model(input_data)
                   ^^^^^^^^^^
NameError: name 'input_data' is not defined
[sel ini berhenti di NameError; notebook tetap lanjut ke sel berikutnya]


Di runtime tanpa GPU, listing empat baris ini meninggalkan dua peringatan di stderr sebelum akhirnya berhenti dengan error.

Peringatan pertama, `FutureWarning`, menyatakan `torch.cuda.amp.autocast(...)` sudah usang dan menyarankan `torch.amp.autocast('cuda', ...)`. Peringatan kedua, `UserWarning: CUDA is not available ... Disabling autocast.`, berarti autocast versi CUDA mematikan dirinya sendiri di runtime tanpa GPU, lalu membiarkan semuanya berjalan di float32 tanpa kabar lain. Di runtime dengan GPU peringatan kedua ini tidak muncul. Autocast-nya aktif, tapi hanya untuk operasi yang berjalan di GPU, dan model kita ada di CPU, jadi hasilnya tetap float32 juga.

Errornya, `NameError: name 'input_data' is not defined`, datang dari placeholder. `model = ...` dan `input_data` sama-sama belum diisi, tapi yang meledak lebih dulu adalah `input_data`, karena Python menghitung argumen sebuah pemanggilan sebelum benar-benar memanggil `Ellipsis`.

Kita isi keduanya, `g` sebagai model dan dua potongan teks validasi sebagai input, lalu menjalankan dua versi berdampingan: autocast buku apa adanya, dan `torch.autocast` dengan `device_type="cpu"`. Autocast di CPU memakai *bfloat16* (bf16), bukan float16 (fp16). Keduanya sama-sama 16 bit, tapi membaginya dengan cara berbeda. bf16 menyimpan 8 bit eksponen seperti float32, sehingga rentangnya sama lebar, tapi hanya 7 bit mantissa, kira-kira dua sampai tiga digit desimal. fp16 punya mantissa yang lebih teliti (10 bit), tapi rentangnya sempit: nilai terbesarnya 65,504. Perbedaan ini akan muncul lagi di tabel format.

In [21]:
model, input_data = g, ft_val[:2]                        # perbaikan #11: GPT-2 (versi Linear) + batch sungguhan
with autocast():                                          # autocast buku (torch.cuda.amp) di mesin CPU
    out_book = model(input_ids=input_data).logits
with torch.autocast("cpu", dtype=torch.bfloat16):         # perbaikan: autocast device-agnostic; dtype CPU = bf16
    out_amp = model(input_ids=input_data).logits
    v_amp, t_amp = eval_gpt2(model), latency_ms(model)
    w_dtype = next(model.parameters()).dtype
print(f"autocast buku (cuda) di CPU   : logits {out_book.dtype} -- tidak ada yang berubah")
print(f"torch.autocast('cpu', bf16)   : logits {out_amp.dtype} | val {v_amp:.3f} (fp32 {base_val:.3f}) "
      f"| {t_amp:.0f} ms (fp32 Linear {latency_ms(model):.0f} ms)")
print(f"bobot di dalam autocast       : {w_dtype} | checkpoint: {nbytes(model)/1e6:.1f} MB (fp32 {fp32_mb:.1f} MB)")


/tmp/ipykernel_584/2993074924.py:2: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():                                          # autocast buku (torch.cuda.amp) di mesin CPU


autocast buku (cuda) di CPU   : logits torch.float32 -- tidak ada yang berubah
torch.autocast('cpu', bf16)   : logits torch.bfloat16 | val 3.775 (fp32 3.762) | 2600 ms (fp32 Linear 682 ms)
bobot di dalam autocast       : torch.float32 | checkpoint: 497.8 MB (fp32 497.8 MB)


Versi buku di CPU tidak mengubah apa pun: logits tetap `float32`, dan satu-satunya jejaknya `FutureWarning` di stderr. Peringatan *CUDA is not available* tidak muncul lagi, kemungkinan karena Python secara bawaan menampilkan peringatan dari baris kode yang sama hanya sekali, dan peringatan itu sudah tampil saat listing tadi dijalankan. Versi CPU bekerja. Logits-nya kini `bfloat16`, dan di run referensi loss-nya 3.766, hanya +0.004. Forward-nya juga lebih cepat di sana, dari 410 ke 180 ms.

Percepatan semacam ini bergantung pada CPU. Ia muncul kalau CPU punya instruksi bf16 seperti `avx512_bf16` atau `amx_bf16` (lihat baris `fitur CPU` dari sel persiapan); di CPU tanpa instruksi itu, bf16 bisa sama lambatnya atau malah lebih lambat daripada float32. Cara mengukurnya juga ikut berperan. Waktu autocast diukur di dalam satu konteks `with torch.autocast(...)` yang panjang, dan selama konteks itu berlangsung autocast bisa menyimpan (*cache*) salinan bf16 dari bobot. Jadi biaya mengubah bobot ke bf16 kemungkinan tidak terhitung di setiap forward, dan angka ini bisa menguntungkan autocast; seberapa besar, tidak diukur di sini.

Sekarang baris terakhir output. Di dalam autocast, **bobot tetap `float32`, dan checkpoint tetap 497.8 MB.** Autocast adalah *mixed-precision compute*: operasi yang ada di daftar autocast dihitung dengan salinan bf16 dari input dan bobotnya. Salinan itu sementara, paling lama bertahan selama konteks autocast seperti yang kita singgung tadi, dan tidak pernah masuk ke `state_dict`; bobot yang disimpan model tetap float32. Tidak ada satu byte pun yang dihemat di checkpoint. Tidak ada pula keputusan per lapisan: daftar operasi yang diubah ke bf16 ditetapkan PyTorch, bukan oleh analisis lapisan mana yang sensitif.

Jadi prosa buku dan listing-nya berbicara tentang dua hal yang berbeda. Mixed-precision quantization yang dimaksud prosa, yaitu lapisan yang kurang kritis di int8 dan lapisan sensitif tetap presisi tinggi, butuh *peta sensitivitas*: seberapa besar kerusakan kalau hanya satu jenis lapisan yang di-int8-kan? Sel berikut membuat peta itu untuk keempat jenis proyeksi di blok GPT-2, masing-masing 12 lapisan, dengan dynamic quantization per-channel `torch.ao`. Lalu kita ambil keputusan dari petanya.

In [22]:
# Mixed-precision quantization yang sesungguhnya: peta sensitivitas per jenis layer, lalu putuskan presisinya
print(f"{'hanya jenis ini yang int8 (torch.ao, dyn per-channel)':<54} {'val loss':>9} {'delta':>7}")
sens = {}
for typ in ("attn.c_attn", "attn.c_proj", "mlp.c_fc", "mlp.c_proj"):
    spec = {f"transformer.h.{i}.{typ}": per_channel_dynamic_qconfig for i in range(12)}
    q = quantize_dynamic(g, spec, dtype=torch.qint8); sens[typ] = eval_gpt2(q); del q; gc.collect()
    print(f"{'12 x ' + typ:<54} {sens[typ]:>9.3f} {sens[typ] - base_val:>+7.3f}")
spec = {f"transformer.h.{i}.{typ}": per_channel_dynamic_qconfig
        for i in range(12) for typ in ("attn.c_attn", "attn.c_proj", "mlp.c_fc")}
q = quantize_dynamic(g, spec, dtype=torch.qint8)
mixed = (eval_gpt2(q), nbytes(q) / 1e6, latency_ms(q)); del q; gc.collect()
full = dyn_rows["per-channel, 48 blok (lm_head fp32)"]
print(f"\nmixed: 36 int8 + 12 mlp.c_proj fp32 : val {mixed[0]:.3f} ({mixed[0] - base_val:+.3f}) | {mixed[1]:.1f} MB | {mixed[2]:.0f} ms")
print(f"semua 48 blok int8                  : val {full[0]:.3f} ({full[0] - base_val:+.3f}) | {full[1]:.1f} MB | {full[2]:.0f} ms")
print(f"jumlah delta per jenis = {sum(v - base_val for v in sens.values()):+.3f} vs delta gabungan = {full[0] - base_val:+.3f}")


hanya jenis ini yang int8 (torch.ao, dyn per-channel)   val loss   delta
12 x attn.c_attn                                           3.785  +0.023
12 x attn.c_proj                                           3.765  +0.003
12 x mlp.c_fc                                              3.794  +0.032
12 x mlp.c_proj                                            3.896  +0.134

mixed: 36 int8 + 12 mlp.c_proj fp32 : val 3.809 (+0.047) | 329.2 MB | 607 ms
semua 48 blok int8                  : val 4.071 (+0.309) | 244.4 MB | 592 ms
jumlah delta per jenis = +0.191 vs delta gabungan = +0.309


Peta sensitivitasnya berat sebelah. Di run referensi, `mlp.c_proj` sendirian menambah +0.134 nats, sekitar 2.5 kali gabungan tiga jenis lainnya (+0.053). Under the hood 1 memberi petunjuk kenapa. Input `mlp.c_proj` adalah keluaran GELU, dan rasio nilai terbesar terhadap mediannya (93 di layer 0, 183 di layer 11) adalah yang tertinggi di antara input blok yang kita intip. (Input `lm_head` jauh lebih ekstrem, tapi ia tidak termasuk blok, dan di peta ini ia tetap float32.) Outlier yang lebih besar berarti scale aktivasi per-tensor yang lebih kasar, dan sensitivitas yang lebih tinggi cocok dengan itu.

Keputusannya mengikuti peta: biarkan 12 `mlp.c_proj` di float32 dan int8-kan sisanya. Kerusakan turun dari sekitar +0.32 ke sekitar +0.08, dengan harga sekitar 85 MB (329.2 lawan 244.4 MB), hampir persis 12 matriks 3,072 x 768 yang disimpan 4 byte per angka alih-alih 1 byte. Di CPU lain, loss hasil `torch.ao` seperti ini bisa bergeser beberapa perseratus nats karena jalur kernelnya berbeda, tapi pergeseran itu jauh lebih kecil daripada penurunan sekitar 0.25 nats ini. Kolom milidetik kedua baris ini tidak bisa dibandingkan langsung, karena angka untuk "semua 48 blok int8" diambil dari pengukuran di sel jauh sebelumnya, bukan diukur ulang di sel ini.

Baris terakhir output adalah yang paling penting untuk keputusan semacam ini. Kalau kerusakan keempat jenis lapisan dijumlahkan, hasilnya sekitar +0.19, padahal kerusakan ketika semuanya di-int8-kan bersama sekitar +0.32. Di eksperimen ini, kerusakan gabungan jauh lebih besar daripada jumlah bagian-bagiannya. Penjelasan yang masuk akal: galat di satu lapisan ikut menggeser input lapisan berikutnya, sehingga galat-galat itu saling memperbesar; notebook ini tidak mengukur mekanisme itu secara langsung. Karena itu peta yang dibuat satu per satu bisa meremehkan harga meng-int8-kan semuanya, dan keputusan mixed precision sebaiknya selalu diuji lagi dalam kombinasi akhirnya.

Masalah yang sama juga bisa dilihat dari sisi lain. Mixed precision di sini sebenarnya menambal granularitas yang salah, yaitu scale aktivasi per-tensor. torchao per-token sudah mencapai +0.005 tanpa menyisakan satu pun proyeksi blok di float32. Dengan resep per-token, di antara 49 proyeksi yang kita uji, yang masih butuh perlakuan khusus tinggal `lm_head`, dan itulah yang kita lakukan saat memberinya quantization bobot saja.

## Hardware-specific considerations

Buku memberi pedoman umum: GPU NVIDIA cocok untuk FP16, sementara *"CPUs often perform better with INT8 quantization because of hardware-accelerated integer operations."* Table 13.1 juga menjanjikan PTQ dengan *"75% storage reduction; 2-4x speedup"*. Kedua klaim itu bergantung pada satu hal yang tidak disebut: instruksi apa yang dimiliki prosesornya.

CPU modern bisa punya instruksi khusus untuk perhitungan presisi rendah. AVX-512 VNNI mempercepat perkalian int8, `avx512_bf16` mempercepat bf16, dan AMX (*Advanced Matrix Extensions*) di Xeon generasi baru menyediakan unit matriks khusus untuk bf16 dan int8 sekaligus. Menurut catatan run referensi, CPU-nya adalah Xeon 2 vCPU dengan AMX (bf16 dan int8) dan AVX-512, jadi ia punya percepatan untuk keduanya. CPU runtime Colab bisa berbeda, dan baris `fitur CPU` dari sel persiapan memberi tahu instruksi mana yang tersedia. Selain empat instruksi khusus tadi (`avx512_vnni`, `avx512_bf16`, `amx_bf16`, `amx_int8`), baris itu menyebut `avx2`, yang dimiliki hampir semua CPU x86 modern, dan `avx512f`, versi 512-bit-nya. Di ujung baris yang sama, `kernel CPU torch` (biasanya `AVX2` atau `AVX512`) menyebut kelompok instruksi vektor terluas yang dipakai torch di CPU itu.

### Under the hood 4: tabel format

Sel berikut (UTH-4) mengukur sepuluh format dalam satu sel dengan kondisi yang sama: ukuran checkpoint, loss, dan latensi, ditambah percepatan terhadap float32 di baris pertama.

In [23]:
# === UNDER THE HOOD (UTH-4): tabel format di CPU yang sedang dipakai -- byte, nats, milidetik ===
fmt = []
def add(label, m, amp=False):
    if amp:
        with torch.autocast("cpu", dtype=torch.bfloat16):
            v, t = eval_gpt2(m), latency_ms(m)
    else:
        v, t = eval_gpt2(m), latency_ms(m)
    fmt.append((label, nbytes(m) / 1e6, v, t))

add("fp32 (Conv1D asli)", gpt2)
add("fp32 (Linear)", g)
add("autocast bf16 (bobot fp32)", g, amp=True)
for label, dt in (("fp16", torch.float16), ("bf16", torch.bfloat16)):
    h = copy.deepcopy(g).to(dt); add(label, h); del h; gc.collect()
q = quantize_dynamic(gpt2, {nn.Linear}, dtype=torch.qint8); add("int8 dynamic {nn.Linear} (resep buku)", q); del q; gc.collect()
q = quantize_dynamic(g, {"transformer.h": per_channel_dynamic_qconfig}, dtype=torch.qint8)
add("int8 dynamic per-channel, 48 blok", q); del q; gc.collect()
q = static_ptq(observers["Histogram"], calib_real); add("int8 static PTQ (Histogram, teks)", q); del q; gc.collect()
q = ao_int8(copy.deepcopy(g), embeddings=False); add("torchao W8A8 per-token, 48 blok", q); del q; gc.collect()
q = ao_int8(copy.deepcopy(g), embeddings=True); add("torchao W8A8 + embedding W8", q); del q; gc.collect()

ref_mb, ref_ms = fmt[0][1], fmt[0][3]
print(f"{'format':<38} {'MB':>7} {'vs fp32':>8} {'val loss':>9} {'delta':>7} {'ms':>5} {'speedup':>8}")
for label, mb, v, t in fmt:
    print(f"{label:<38} {mb:>7.1f} {mb / ref_mb - 1:>+8.0%} {v:>9.3f} {v - base_val:>+7.3f} {t:>5.0f} {ref_ms / t:>7.2f}x")


format                                      MB  vs fp32  val loss   delta    ms  speedup
fp32 (Conv1D asli)                       497.8      +0%     3.762  +0.000   688    1.00x
fp32 (Linear)                            497.8      +0%     3.762  +0.000   676    1.02x
autocast bf16 (bobot fp32)               497.8      +0%     3.775  +0.013  2583    0.27x
fp16                                     248.9     -50%     3.763  +0.001  6254    0.11x
bf16                                     248.9     -50%     3.770  +0.008  2459    0.28x
int8 dynamic {nn.Linear} (resep buku)    536.4      +8%     8.995  +5.233   644    1.07x
int8 dynamic per-channel, 48 blok        244.4     -51%     4.071  +0.309   570    1.21x
int8 static PTQ (Histogram, teks)        244.4     -51%     4.760  +0.998   538    1.28x
torchao W8A8 per-token, 48 blok          243.5     -51%     3.767  +0.005 15744    0.04x
torchao W8A8 + embedding W8              127.9     -74%     3.780  +0.018 16605    0.04x


Tabel ini paling mudah dibaca kolom demi kolom, karena tidak semua kolom berlaku umum. Kolom MB dan loss adalah sifat formatnya, dan di CPU lain biasanya hanya bergeser sedikit. Kolom milidetik dan percepatan adalah sifat CPU yang mengukurnya. Jadi kita baca dulu tabel yang tersimpan, lalu apa yang perlu diperhatikan saat tabel ini dibuat ulang di Colab.

Sebelum membaca kolom milidetik, ingat seberapa goyah angkanya. Di run referensi, GPT-2 float32 asli, `gpt2`, terbaca 493, 414, dan 329 ms di tiga sel berbeda, dan versi Linear-nya, `g`, terbaca 378, 410, dan 432 ms. Penyebabnya tidak kita ukur; kemungkinan CPU 2 vCPU itu juga sedang melayani pekerjaan lain. Kolom percepatan di sini pun dihitung terhadap satu pengukuran float32 saja, di baris pertama. Karena itu milidetik hanya kita bandingkan di dalam satu sel, dan hanya selisih besar yang kita anggap sinyal. Di tabel ini, sinyal itu datang dari bf16, autocast, dan varian int8 yang menyasar 48 blok; fp16, resep buku (1.21x), dan varian dengan embedding int8 belum bisa dibedakan dengan yakin dari float32. Aturan yang sama berlaku saat membaca tabel yang dihasilkan di Colab.

Di CPU run referensi, yang menurut catatannya punya AMX untuk bf16 dan int8, format tercepat adalah **bf16**: sekitar 3 kali lebih cepat daripada float32, dengan ukuran separuhnya dan loss yang hampir tidak berubah (+0.011). Varian int8 yang menyasar 48 blok hanya sekitar 1.6 sampai 1.9 kali lebih cepat, dan urutan kerusakannya sejalan dengan cara scale aktivasi dihitung: beku satu per lapisan (PTQ statis, sekitar 1 nats, yang juga membulatkan output setiap lapisan), dihitung ulang tapi satu untuk seluruh tensor (`torch.ao` dinamis, sekitar 0.3), atau satu per token (torchao, praktis nol). fp16 kebalikan bf16: ukurannya sama-sama separuh dan loss-nya malah sedikit lebih baik, kemungkinan berkat mantissa yang lebih teliti, tapi tidak lebih cepat sama sekali, tampaknya karena perkalian matriks fp16 tidak dipercepat di CPU itu. Jadi di CPU tersebut, klaim buku bahwa *"CPUs often perform better with INT8"* hanya separuh benar: varian int8 yang menyasar 48 blok memang lebih cepat daripada float32 dan fp16, tapi bukan pemenangnya. Yang menang adalah format yang dipercepat oleh hardware yang benar-benar ada.

Klaim Table 13.1, *"75% storage reduction; 2-4x speedup"*, tidak pernah terjadi bersamaan di sini. Pengurangan 74% hanya dicapai varian yang ikut meng-int8-kan embedding, dan varian itu tidak lebih cepat daripada float32 (0.89x). Varian int8 yang cepat hanya memotong 51% byte, karena embedding 38.6 juta parameter masih float32, dan tidak satu pun mencapai 2x. Resep buku sendiri menempati baris terburuk: satu-satunya format yang lebih besar daripada float32 (+8%), dengan kerusakan +5.233 nats untuk percepatan 1.21x. Percepatan sekecil itu belum bisa dibedakan dengan yakin dari derau, dan karena resep ini hanya menyentuh `lm_head`, hitungan MAC di awal bab memang membatasinya di sekitar 1.45x.

Di CPU tanpa instruksi bf16, urutannya bisa berbalik: bf16 dan autocast bisa tidak lebih cepat, atau malah lebih lambat, daripada float32. Di keluarga int8, varian `torch.ao` bisa menjadi yang tercepat, sedangkan torchao W8A8 bisa ikut lebih lambat daripada float32. Jadi, saat tabel ini dijalankan ulang, baca kolom milidetiknya bersama baris `fitur CPU` dari sel persiapan. Kalau `avx512_bf16` dan `amx_bf16` sama-sama "tidak", kemungkinan besar bf16 tidak akan menjadi pemenang. Untuk int8, lihat `avx512_vnni` dan `amx_int8`; kalau keduanya "tidak", kernel int8 berjalan dengan instruksi yang lebih umum seperti AVX2, jadi wajar kalau percepatannya jauh berbeda dari run referensi, bahkan berubah menjadi perlambatan.

## Comparing quantization strategies

Table 13.1 di buku menilai empat strategi menurut akurasi, kompleksitas, performa, dan kebutuhan sumber daya. Ada dua hal yang perlu diluruskan sebelum membandingkan. Pertama, pembagiannya bertabrakan dengan prosa buku sendiri, yang menyebut PTQ *"can be implemented as either static (with calibration) or dynamic"*. Jadi dynamic bukan saingan PTQ, melainkan salah satu bentuknya; di bab ini, semua yang bukan QAT adalah PTQ. Kedua, angka yang ada di tabel itu (75%, 2-4x) disajikan tanpa menyebut model, data, atau hardware-nya.

Tabel berikut adalah versi yang diisi dengan pengukuran bab ini. Semua angkanya dari run referensi, pada GPT-2 124M kecuali baris QAT yang berasal dari model mini. Kolom kecepatan diambil dari tabel Under the hood 4 dan hanya berlaku untuk CPU itu. Tanda -- berarti angkanya tidak diukur di tabel tersebut, atau, untuk QAT, tidak sebanding karena modelnya lain.

| Strategi (bab ini) | Selisih loss | Yang benar-benar dibutuhkan | Byte | Kecepatan (UTH-4) |
|---|---|---|---|---|
| PTQ statis, `torch.ao` (Histogram, teks) | +1.000 | Conv1D ke Linear, stub di 48 Linear, kalibrasi | -51% | 1.88x |
| PTQ dinamis, `torch.ao` per-channel | +0.324 | Conv1D ke Linear, qconfig per nama modul | -51% | 1.56x |
| PTQ dinamis, torchao per-token | +0.005 | Conv1D ke Linear, satu panggilan `quantize_` | -51% | 1.74x |
| ditambah embedding int8 bobot saja | +0.018 | ikatan *tied* dipasang ulang manual | -74% | 0.89x |
| Mixed per lapisan (`mlp.c_proj` float32) | +0.077 | peta sensitivitas per jenis lapisan | -34% | -- |
| QAT (model mini, untung terhadap kontrol) | 4 bit: 0.002, 3 bit: 0.009, 2 bit: 0.685 | stub, loop training, kelompok kontrol | -- | -- |

Dibaca dengan tabel ini, tiga penilaian Table 13.1 berbalik. Dynamic dinilai *"good for RNNs"* dengan *"slower compute"*; di sini dynamic per-token justru varian int8 yang paling akurat, dengan kecepatan sekelas statis di CPU tadi. PTQ dinilai *"Low; minimal setup"*; di GPT-2, varian PTQ yang sederhana sekaligus akurat tetap butuh pengetahuan yang tidak ada di buku, yaitu mengganti `Conv1D` dan tahu lapisan mana yang aktivasinya tidak boleh di-quantize, sedangkan resep yang paling sederhana, resep buku, justru paling merusak. QAT dinilai *"Highest; best for sub-8-bit"*. Di model mini kita (satu seed), keunggulan QAT di int8 serta di 4 dan 3 bit tidak bisa dibedakan dari training tambahan, yang juga bisa didapat tanpa QAT. Baru di 2 bit, pada percobaan yang hanya meng-quantize bobot, QAT jelas unggul.

## Combining quantization with other optimization techniques

Quantization bisa digabung dengan teknik kompresi lain. Kombinasi pertama di buku adalah *pruning* lalu quantization. Pruning membuang bobot yang dianggap tidak penting; versi yang dipakai buku, `l1_unstructured` dengan `amount=0.5`, menjadikan nol separuh bobot dengan nilai absolut terkecil di setiap lapisan. Sesudahnya quantization memurahkan sisanya, *"which further decreases the model size"*, kata buku.

Pembaca Chapter 12 akan mengenali listing ini. Kita jalankan sekali lagi apa adanya.

In [24]:
%%boleh_gagal
import torch
import torch.nn.utils.prune as prune
import torch.quantization as quant
# Step 1: Prune the model
model = ... # Pre-trained LLM model
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Linear):
        prune.l1_unstructured(module, name='weight', amount=0.5)
        # Prune 50% of the weights
        prune.remove(module, 'weight')
# Step 2: Apply dynamic quantization to the pruned model
quantized_model = quant.quantize_dynamic(
    model, {torch.nn.Linear}, dtype=torch.qint8 # Convert to INT8 precision
)
# Check size reduction
print("Original model size:", torch.cuda.memory_allocated())
print("Quantized model size:", torch.cuda.memory_allocated())


Traceback (most recent call last):
  File "<listing-8>", line 6, in <module>
    for name, module in model.named_modules():
                        ^^^^^^^^^^^^^^^^^^^
AttributeError: 'ellipsis' object has no attribute 'named_modules'
[sel ini berhenti di AttributeError; notebook tetap lanjut ke sel berikutnya]


Placeholder yang sama, error yang serupa. Kali ini yang pertama kali menyentuh `model` adalah loop pruning, sehingga pesannya menjadi `AttributeError: 'ellipsis' object has no attribute 'named_modules'`. Seandainya placeholder-nya diisi GPT-2, kita sudah bisa menduga sisanya dari bagian-bagian sebelumnya: sasaran `nn.Linear` yang di GPT-2 hanya berarti `lm_head`, dan `torch.cuda.memory_allocated()` yang mencetak 0 sebagai "ukuran model".

Pertanyaan yang lebih menarik belum terjawab. Kalau pruning menyasar 48 proyeksi blok, dan quantization memakai resep terbaik bab ini (torchao per-token ditambah embedding int8), apa yang sebenarnya didapat dari pruning? Sel berikut mencoba tiga tingkat *sparsity*, yaitu persentase bobot blok yang dinolkan: 0%, 30%, dan 50%. Untuk setiap tingkat kita ukur ukuran dan loss model float32, lalu ukuran dan loss sesudah quantization. Satu kolom tambahan mengukur checkpoint int8 setelah dikompres dengan zlib, kompresor serbaguna dengan algoritma yang sama seperti file zip. Kolom ini memperlihatkan berapa byte yang *bisa* dihemat dari nol-nol hasil pruning kalau file-nya dikompres.

In [25]:
import torch.nn.utils.prune as prune
print(f"{'sparsity blok':>13} {'fp32 MB':>8} {'fp32 loss':>10} {'int8 MB':>8} {'int8 zlib MB':>13} {'int8 loss':>10}")
prune_rows = {}
for amount in (0.0, 0.3, 0.5):
    m = copy.deepcopy(g)
    if amount:
        for n, mod in m.named_modules():
            if isinstance(mod, nn.Linear) and n.startswith("transformer.h"):    # perbaikan: sasaran 48 blok, tie utuh
                prune.l1_unstructured(mod, name='weight', amount=amount); prune.remove(mod, 'weight')
    lf, mbf = eval_gpt2(m), nbytes(m) / 1e6
    ao_int8(m, embeddings=True)
    torch.save(m.state_dict(), "_z.pt"); raw = open("_z.pt", "rb").read(); os.remove("_z.pt")
    prune_rows[amount] = (mbf, lf, len(raw) / 1e6, len(zlib.compress(raw, 6)) / 1e6, eval_gpt2(m))
    print(f"{amount:>13.0%} {mbf:>8.1f} {lf:>10.3f} {prune_rows[amount][2]:>8.1f} {prune_rows[amount][3]:>13.1f} {prune_rows[amount][4]:>10.3f}")
    del m, raw; gc.collect()


sparsity blok  fp32 MB  fp32 loss  int8 MB  int8 zlib MB  int8 loss
           0%    497.8      3.762    127.9         114.6      3.780
          30%    497.8      4.167    127.9         105.3      4.201
          50%    497.8      6.266    127.9          93.5      6.298


Kolom-kolom yang tidak berubah justru temuannya. **Pruning tidak menghemat satu byte pun dalam penyimpanan biasa**: di run referensi, checkpoint float32 tetap 497.8 MB dan checkpoint int8 tetap 127.9 MB di setiap tingkat sparsity. Bobot yang dinolkan tetap disimpan sebagai satu elemen, hanya saja nilainya nol. Kalimat buku *"which further decreases the model size"* baru benar kalau file-nya dikompres, atau disimpan dalam format *sparse* yang hanya mencatat elemen yang bukan nol.

Kolom zlib memperlihatkan berapa yang bisa dipanen: dari 114.6 ke 93.5 MB di sparsity 50%, sekitar 18% lebih kecil. Harganya mahal. Loss model int8 naik dari 3.780 ke 6.298, +2.52 nats, yang berarti peluang untuk token yang benar tinggal kurang dari sepersepuluhnya. Bandingkan dengan quantization sendirian, yang memangkas 74% byte untuk +0.018 nats. Dalam hitungan byte per nats, pruning tak terstruktur seperti ini tidak sepadan. Arah yang lebih menjanjikan adalah pruning *terstruktur*, yang membuang neuron, head, atau layer utuh sehingga matriksnya benar-benar mengecil, atau runtime yang memang bisa memanfaatkan sparsity; keduanya tidak diuji di sini.

Kolom loss int8 juga memperlihatkan bahwa int8 hanya menambah kerusakan kecil di atas model yang sudah dipangkas, sekitar +0.02 sampai +0.03 nats, jauh lebih kecil daripada kerusakan akibat pruning-nya sendiri.

Kombinasi kedua adalah *knowledge distillation* (KD) lalu quantization. Model kecil, disebut *student*, dilatih meniru model besar, *teacher*. Yang ditiru bukan hanya token yang benar, tapi seluruh sebaran peluang teacher di setiap posisi. Kalau teacher memberi "cat" 60% dan "dog" 30%, student ikut belajar bahwa "dog" juga jawaban yang masuk akal. Sebaran itu biasanya dihaluskan dulu dengan *temperature* T: logits dibagi T sebelum softmax, sehingga peluang yang kecil ikut terlihat. Sesudah student terlatih, ia di-quantize.

Buku menyajikannya dalam tujuh langkah. Langkah 1 sampai 5, yaitu definisi model, fungsi loss distilasi, dan loop training, digabung di satu sel, dengan indentasi loop yang dipulihkan karena rusak oleh pemenggalan baris di PDF. Placeholder-nya kini dua: `teacher_model` dan `student_model`.

In [26]:
%%boleh_gagal
import torch
import torch.nn.functional as F
teacher_model = ... # Larger, fully trained model
student_model = ... # Smaller model to be trained through distillation
def distillation_loss(
    student_outputs, teacher_outputs, temperature=2.0
):
    teacher_probs = F.softmax(
        teacher_outputs / temperature, dim=1)
    student_probs = F.log_softmax(
        student_outputs / temperature, dim=1)
    return F.kl_div(student_probs, teacher_probs,
                    reduction='batchmean')
optimizer = torch.optim.Adam(student_model.parameters(),
                             lr=1e-4)
for batch in train_loader:
    inputs, _ = batch
    optimizer.zero_grad()
    teacher_outputs = teacher_model(inputs)
    student_outputs = student_model(inputs)
    loss = distillation_loss(student_outputs, teacher_outputs)
    loss.backward()
    optimizer.step()


Traceback (most recent call last):
  File "<listing-9>", line 14, in <module>
    optimizer = torch.optim.Adam(student_model.parameters(),
                                 ^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'ellipsis' object has no attribute 'parameters'
[sel ini berhenti di AttributeError; notebook tetap lanjut ke sel berikutnya]


Di listing distilasi, yang pertama kali menyentuh placeholder adalah pembuatan optimizer dari `student_model.parameters()`, jadi errornya `AttributeError: 'ellipsis' object has no attribute 'parameters'`. Fungsi `distillation_loss` sudah sempat terdefinisi sebelum baris itu dan tetap tersedia untuk sel berikutnya; sebentar lagi kita periksa isinya.

Kita isi placeholder-nya. Teacher adalah GPT-2 124M. Student adalah GPT-2 6 layer yang mewarisi layer genap teacher (0, 2, 4, 6, 8, dan 10), mirip ide DistilGPT-2 yang mengambil satu dari setiap dua layer teacher sebagai titik awal. `train_loader` diisi potongan teks wikitext-2 bagian train, dan loop-nya dijalankan persis seperti di buku.

In [27]:
%%boleh_gagal
from torch.utils.data import DataLoader, TensorDataset
def make_student():                                        # student 6 layer: lapis genap teacher (pola DistilGPT-2)
    s = copy.deepcopy(gpt2)
    s.transformer.h = nn.ModuleList([s.transformer.h[i] for i in (0, 2, 4, 6, 8, 10)]); s.config.n_layer = 6
    for new_i, blk in enumerate(s.transformer.h):
        for mod in blk.modules():
            if hasattr(mod, "layer_idx"): mod.layer_idx = new_i     # indeks KV-cache harus 0..5
    return s
teacher_model = gpt2                                       # perbaikan #14: placeholder diisi model sungguhan
student_model = make_student()
train_loader = DataLoader(TensorDataset(ft_train, ft_train), batch_size=2)
optimizer = torch.optim.Adam(student_model.parameters(), lr=1e-4)
for batch in train_loader:                                 # loop persis buku
    inputs, _ = batch
    optimizer.zero_grad()
    teacher_outputs = teacher_model(inputs)
    student_outputs = student_model(inputs)
    loss = distillation_loss(student_outputs, teacher_outputs)
    loss.backward()
    optimizer.step()


Traceback (most recent call last):
  File "<listing-10>", line 18, in <module>
    loss = distillation_loss(student_outputs, teacher_outputs)
  File "<listing-9>", line 9, in distillation_loss
    teacher_outputs / temperature, dim=1)
    ~~~~~~~~~~~~~~~~^~~~~~~~~~~~~
TypeError: unsupported operand type(s) for /: 'CausalLMOutputWithCrossAttentions' and 'float'
[sel ini berhenti di TypeError; notebook tetap lanjut ke sel berikutnya]


Kali ini kedua model sempat dipanggil, tapi loop-nya tersandung di perhitungan loss. Di run referensi errornya `TypeError: unsupported operand type(s) for /: 'CausalLMOutputWithCrossAttentions' and 'float'`. Model Hugging Face tidak mengembalikan tensor logits, melainkan sebuah objek output yang berisi logits beserta hal lain, dan `distillation_loss` mencoba membagi objek itu dengan temperature. Perbaikan minimalnya: ambil `.logits`.

Dengan itu loop-nya berjalan. Tapi berjalan belum berarti benar. Fungsi loss buku memanggil `softmax(..., dim=1)`, padahal logits model bahasa berbentuk (batch, posisi, kosakata), di sini (2, 128, 50257). Sumbu 1 adalah *posisi*, bukan kosakata. Untuk setiap token di kosakata, fungsi itu menormalkan skornya di sepanjang 128 posisi kalimat. Student jadinya diminta meniru *di posisi mana* teacher paling yakin akan kata "the", bukan *kata apa* yang diprediksi teacher di setiap posisi. Pada model klasifikasi, logits berbentuk (batch, kelas) dan `dim=1` memang sumbu kelas; tampaknya resep klasifikasi itu tersalin begitu saja ke LLM.

Sel berikut mengukur akibatnya. Dua student identik dilatih 30 langkah dengan Adam dan learning rate 1e-4 seperti di buku: satu dengan fungsi loss buku yang hanya diberi `.logits`, satu dengan versi yang diperbaiki. Versi perbaikan menormalkan di sumbu kosakata, merata-ratakan per token, lalu mengalikan hasilnya dengan T^2. Faktor T^2 ini koreksi dari paper distilasi Hinton dan kawan-kawan: membagi logits dengan T mengecilkan gradien kira-kira sebesar 1/T^2, dan faktor itu mengembalikannya ke skala semula.

Baris-baris awal sel berikut juga melepas sisa loop yang gagal dan `g`, GPT-2 versi Linear yang tidak dipakai lagi, supaya RAM cukup untuk melatih dua student. Akibatnya sel itu tidak bisa diulang sendirian: kalau dijalankan lagi, ia berhenti dengan `NameError` karena nama-nama itu sudah terhapus. Untuk mengulang bagian ini, jalankan ulang notebook dari sel persiapan.

In [28]:
def kd_book(s_logits, t_logits):                          # fungsi buku apa adanya, hanya diberi .logits (perbaikan #15)
    return distillation_loss(s_logits, t_logits)
def kd_fixed(s_logits, t_logits, T=2.0):                   # perbaikan #16: softmax di sumbu vocab, rata-rata per token, x T^2
    V = s_logits.size(-1)
    return F.kl_div(F.log_softmax(s_logits.reshape(-1, V) / T, dim=-1),
                    F.softmax(t_logits.reshape(-1, V) / T, dim=-1), reduction="batchmean") * T * T
del student_model, optimizer, teacher_outputs, student_outputs, inputs, batch   # sisa sel gagal (graf gradien student ikut tertahan)
del g, model, input_data, out_book, out_amp; gc.collect()                         # knob memori: GPT-2 versi Linear tak dipakai lagi
s_tmp = make_student(); student_init = eval_gpt2(s_tmp); n_student = sum(p.numel() for p in s_tmp.parameters()); del s_tmp
print(f"teacher 12L: val {base_val:.3f} | student 6L awal (lapis genap): val {student_init:.3f} | {n_student/1e6:.1f}M parameter")
students = {}
for label, lossfn in (("buku: softmax dim=1", kd_book), ("diperbaiki: dim=-1, T^2", kd_fixed)):
    torch.manual_seed(0)
    st = make_student(); opt = torch.optim.Adam(st.parameters(), lr=1e-4); st.train()
    t0 = time.time(); first = last = None
    for s in range(30):
        x = ft_train[(s * 2) % (len(ft_train) - 2):(s * 2) % (len(ft_train) - 2) + 2]
        with torch.no_grad():
            t_logits = gpt2(input_ids=x).logits             # teacher tanpa graf gradien
        loss = lossfn(st(input_ids=x).logits, t_logits)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        first = loss.item() if first is None else first; last = loss.item()
    del opt
    print(f"{label:<26}: loss KD {first:.4g} -> {last:.4g} | student val CE {student_init:.3f} -> {eval_gpt2(st):.3f} "
          f"| {time.time() - t0:.0f}s")
    if label.startswith("diperbaiki"): students[label] = st                    # knob memori: hanya student yang benar disimpan
    else: del st; gc.collect()


teacher 12L: val 3.762 | student 6L awal (lapis genap): val 5.438 | 81.9M parameter
buku: softmax dim=1       : loss KD 6.775e+05 -> 2.821e+05 | student val CE 5.438 -> 6.495 | 102s
diperbaiki: dim=-1, T^2   : loss KD 3.045 -> 1.29 | student val CE 5.438 -> 4.902 | 96s


Sebelum dilatih, student 6 layer sudah jauh lebih buruk daripada teacher: di run referensi loss-nya 5.438 lawan 3.762. Membuang separuh layer langsung menambah sekitar 1.7 nats. Modelnya kini 81.9 juta parameter, bukan separuh dari 124.4 juta, karena embedding `wte` (38.6 juta) dan `wpe` tetap utuh; yang dibuang hanya enam blok, masing-masing sekitar 7.1 juta parameter.

Fungsi loss buku memperlihatkan jebakan yang licin. Loss KD-nya turun meyakinkan, dari sekitar 6.8 x 10^5 ke 2.8 x 10^5, sehingga optimizer tampak bekerja keras. Tapi student-nya **memburuk 1.06 nats** (5.438 ke 6.494) dalam 30 langkah itu. Ia memang mengoptimalkan sesuatu, hanya saja sesuatu di sumbu yang salah, dan hasilnya menjauh dari bahasa. Besaran ratusan ribu itu sendiri juga petunjuk. `reduction='batchmean'` hanya membagi dengan ukuran batch (2), sehingga KL dijumlahkan atas 128 x 50,257 entri tanpa dirata-ratakan per token. Kalau loss KD sebesar itu dijumlahkan dengan loss lain, misalnya cross-entropy terhadap token yang benar yang besarnya hanya beberapa nats, loss lain itu praktis tidak ikut berpengaruh.

Versi yang diperbaiki membawa student yang sama turun 0.54 nats (5.438 ke 4.902) dalam 30 langkah yang sama. Perbedaan utamanya kemungkinan besar sumbu softmax, walaupun versi perbaikan juga mengubah normalisasi per token dan menambahkan faktor T^2, dan percobaan ini tidak memisahkan ketiganya. Yang jelas, tidak ada error yang memperingatkan bahwa loss buku mengoptimalkan hal yang salah. Loss KD yang turun tidak berarti student-nya membaik; hanya penggaris kita yang menunjukkan arah sebenarnya.

Langkah 6 dan 7 buku, yaitu quantize student lalu ukur ukurannya, kita jalankan persis seperti dicetak pada student yang sudah disuling dengan benar.

In [29]:
# Langkah 6-7 buku, persis -- pada student yang disuling dengan benar
student_model = students["diperbaiki: dim=-1, T^2"]
quantized_student_model = quant.quantize_dynamic(
    student_model, {torch.nn.Linear}, dtype=torch.qint8
)
print("Quantized student model size:",
      torch.cuda.memory_allocated())

print("--- diukur ---")
print(f"student fp32                        : {nbytes(student_model)/1e6:6.1f} MB | val {eval_gpt2(student_model):.3f}")
print(f"langkah 6 buku ({{nn.Linear}})        : {nbytes(quantized_student_model)/1e6:6.1f} MB | val {eval_gpt2(quantized_student_model):.3f}")
del quantized_student_model; gc.collect()
s8 = copy.deepcopy(student_model); conv1d_to_linear(s8); ao_int8(s8, embeddings=True)
print(f"diperbaiki: torchao W8A8 + emb W8    : {nbytes(s8)/1e6:6.1f} MB | val {eval_gpt2(s8):.3f} | {latency_ms(s8):.0f} ms "
      f"(teacher fp32: {fp32_mb:.1f} MB, val {base_val:.3f}, {latency_ms(gpt2):.0f} ms)")


Quantized student model size: 0
--- diukur ---
student fp32                        :  327.7 MB | val 4.902
langkah 6 buku ({nn.Linear})        :  366.3 MB | val 6.349
diperbaiki: torchao W8A8 + emb W8    :   85.0 MB | val 4.909 | 8369 ms (teacher fp32: 497.8 MB, val 3.762, 679 ms)


Langkah penutup listing KD mengulang kesalahan-kesalahan yang sudah kita kenal. Langkah 6 memakai `quant.quantize_dynamic`, padahal listing KD tidak pernah meng-import `quant`; di notebook ini ia berjalan karena `quant` sudah di-import di sel fondasi. Ukuran yang dicetak `torch.cuda.memory_allocated()` kembali 0. Dan `{nn.Linear}` pada student GPT-2 sekali lagi hanya berarti `lm_head`: di run referensi student *membesar* dari 327.7 ke 366.3 MB, dan loss hasil distilasi yang tadi dibangun dengan susah payah rusak dari 4.902 ke 6.349.

Resep bab ini pada student yang sama memberi 85.0 MB dengan loss 4.917, hanya +0.015 dari student float32. Jadi dari teacher 497.8 MB, rantai distilasi lalu quantization berakhir di 85 MB, hampir enam kali lebih kecil, dengan harga 1.16 nats. Hampir seluruh harga itu dibayar saat membuang enam layer, yang baru sebagian dipulihkan oleh 30 langkah distilasi; quantization hanya menambah 0.015.

Kolom latensi menutup bab dengan pengingat yang sama seperti sebelumnya. Di run referensi, student int8 hampir tidak lebih cepat daripada teacher float32 (302 lawan 329 ms, diukur di sel yang sama); di CPU tanpa instruksi int8 khusus ia bahkan bisa lebih lambat. Sama seperti di bagian torchao, embedding int8 yang harus di-dequantize di setiap forward kemungkinan memakan sebagian besar waktu yang dihemat dengan membuang enam layer. Sekali lagi, model yang lebih kecil tidak otomatis lebih cepat.

## Summary

Bab ini mengambil janji quantization, model yang jauh lebih kecil dan beberapa kali lebih cepat, lalu mengujinya di GPT-2 124M sungguhan dengan satu penggaris yang sama untuk semua varian: loss validasi wikitext-2 dalam nats.

Prinsip dasarnya sederhana. Quantization membulatkan setiap angka ke deretan nilai yang lebih jarang, dan seberapa jarang ditentukan oleh rentang yang harus dijangkau satu scale. Karena itu kerusakan tidak hanya ditentukan oleh jumlah bit, tapi juga oleh di mana nilai-nilai ekstrem berada. Di GPT-2, Under the hood 1 menemukan bahwa bobot int8 dengan scale per baris hampir tidak merusak. Kerusakan terbesar datang dari aktivasi, terutama dimensi 496 di keluaran `ln_f` yang nilai absolutnya mencapai 330, padahal median nilai absolut seluruh input `lm_head` hanya 0.31. Dengan satu scale untuk seluruh tensor, sedikitnya separuh input `lm_head` dibulatkan menjadi nol.

Dari diagnosis itu, resepnya mengikuti. Ganti 48 `Conv1D` dengan `nn.Linear` supaya API quantization bisa melihatnya. Pakai bobot per-channel dan aktivasi per-token di blok, lewat torchao, pengganti `torch.ao.quantization` yang ditunjuk pesan deprecation torch sendiri. Jangan meng-quantize aktivasi di `lm_head`; cukup bobot embedding-nya. Di run referensi hasilnya 127.9 MB, 74% lebih kecil, dengan loss hanya +0.018 nats. Sebagai perbandingan, resep buku, `quantize_dynamic` dengan `{nn.Linear}`, hanya menyentuh `lm_head`, membuat checkpoint membesar ke 536 MB, dan merusak loss ke hampir 9.

Static quantization butuh lebih banyak daripada yang ditunjukkan listing buku: `QuantStub` dan `DeQuantStub` untuk menandai batas float dan int8, qconfig yang tidak menyentuh embedding, dan terutama kalibrasi. Tanpa kalibrasi semua scale jatuh ke 1.0. Dengan kalibrasi, observer Histogram jauh lebih baik daripada MinMax, dan token acak hampir sama baiknya dengan teks asli, kemungkinan karena outlier GPT-2 muncul di dimensi yang tetap. Meski begitu, PTQ statis terbaik masih kehilangan sekitar 1 nats, jauh di belakang PTQ dinamis.

Diterapkan pada GPT-2 mini, QAT versi buku hanya mengganti `lm_head` dengan versi QAT, tidak menyentuh kedelapan `Conv1D` di bloknya, lalu gagal di `convert`. Dengan kelompok kontrol yang dilatih sama lama, keunggulan QAT di 8, 4, dan 3 bit tidak bisa dibedakan dari efek training tambahan (satu seed). Di model mini kita, QAT baru jelas menguntungkan di 2 bit, saat setiap baris hanya punya tiga nilai.

"Mixed precision" di listing buku adalah autocast, yaitu perhitungan dengan presisi campuran yang tidak mengecilkan satu byte pun. Mixed-precision quantization yang dimaksud prosa buku butuh peta sensitivitas, yang di GPT-2 menunjuk `mlp.c_proj`. Peta itu juga perlu diuji dalam kombinasi akhirnya, karena di eksperimen kita kerusakan antarlapisan tidak sekadar dijumlahkan.

Byte dan milidetik adalah dua anggaran yang berbeda. Ukuran dan loss adalah sifat format; kecepatan bergantung pada instruksi yang dimiliki CPU. Di CPU run referensi, yang punya AMX, bf16 justru format tercepat, dan tidak ada varian yang sekaligus memangkas sekitar 75% byte dan dua kali lebih cepat. Di CPU lain urutan kecepatannya bisa berbeda, dan baris `fitur CPU` di awal notebook membantu membacanya.

Untuk kombinasi, pruning tak terstruktur tidak mengurangi satu byte pun pada checkpoint biasa, dan fungsi loss distilasi buku, dengan `softmax(dim=1)` di sumbu posisi, membuat student lebih buruk walaupun loss KD-nya turun. Dengan loss yang diperbaiki dan resep int8 bab ini, student 6 layer menjadi 85 MB, hampir enam kali lebih kecil daripada teacher, tapi loss-nya 4.917 lawan 3.762. Hampir seluruh selisih itu berasal dari membuang enam layer, yang baru sebagian dipulihkan oleh 30 langkah distilasi; quantization sendiri hanya menambah 0.015. Model yang lebih kecil belum tentu cukup baik, jadi ukuran dan kualitas perlu diukur terpisah.

Di sepanjang jalan, banyak listing buku perlu diperbaiki sebelum bisa dijalankan pada model sungguhan: placeholder `...` yang ikut dieksekusi, `torch.cuda.memory_allocated()` yang dipakai untuk mengukur model di CPU, kurung yang tidak ditutup, nama yang tidak pernah didefinisikan, dan `{nn.Linear}` yang di GPT-2 hanya berarti satu lapisan. Akarnya sama: listing buku ditulis untuk model sederhana, sedangkan GPT-2 punya embedding, outlier, dan `Conv1D`.

### What comes next

Bab ini mengukur semuanya dengan satu angka, loss validasi, dan itu cukup untuk membandingkan format. Tapi apakah model yang ter-quantisasi masih menulis, memahami, dan berdialog sebaik aslinya? Chapter 14 membahas evaluasi LLM, dengan metrik untuk text generation, language understanding, dan dialogue systems. Di sana model-model ringkas seperti di bab ini bisa diuji pada tugas, bukan hanya pada loss.